In [ ]:
#!/usr/bin/env python3
"""
FUSE-JIT reconstruction for JIT-Defect-Extended
=================================================

This script rebuilds the methodology described in the supplied FUSE-JIT paper
on the 21-project JIT-Defect4J/JIT-Defect-Extended subjects.

Design goals
------------
1. Commit-level experimental unit.
2. Labels are taken ONLY from JIT-Defect-Extended (dataset/buggy, dataset/no_buggy).
3. Old/new Java source pairs, author timestamps, commit messages, and process
   metrics are reconstructed from the corresponding Git repository.
4. Four views are fused (the original three FUSE-JIT views, plus a fourth added
   in the second accuracy-improvement pass below):
      - AST-based syntactic changes -> Word2Vec (100d, window=10), length 50
      - hybrid program-graph changes -> Node2Vec-style embeddings (100d,
        context window=5), length 50
      - 14 process/expert metrics: NS, ND, NF, Entropy, FIX, NUC, LA, LD, LT,
        NDEV, AGE, EXP, REXP, SEXP
      - a pretrained code-model (CodeBERT-base) embedding of each commit's diff
5. Syntax maps preserve the original commit-level multi-file max aggregation.
   Cached per-graph Node2Vec vectors are converted to rotation-invariant structural
   quantities before coherent multi-file aggregation, avoiding arbitrary-axis mixing.
6. Whole-dataset evaluation over all 21 projects: one random 8:1:1
   train/validation/test split (seed 42), following the benchmark protocol used by
   JIT-CF/JIT-Fine-style experiments rather than training a separate model per project.
7. Word2Vec vocabulary/weights, graph categorical vocabulary, process normalization,
   threshold selection, and class weights are fitted/derived from training data only.
8. Architecture retains three CNN branches + two feature-wise sigmoid gates, with
   normalization added for stable optimization.
9. 30 independent runs, max 100 epochs, batch 64, AdamW lr=1e-3, weight decay 1e-4,
   gradient clipping, validation scheduling and early stopping.
10. Reports Precision, Recall, F1, AUC, MCC, Accuracy, Balanced Accuracy, PofB20,
    R@20%E, E@20%R, Popt, Top5, Top10 and IFA on the untouched global test split.

Important reproducibility note
------------------------------
The paper specifies the methodology but does not publish the original FUSE-JIT
source code or every low-level implementation choice (e.g., exact GumTree
matching thresholds, Node2Vec p/q/walk counts, node/relation embedding widths,
or validation threshold-search grid). The defaults below implement every
explicitly documented choice and make the underspecified choices deterministic
and configurable. This is therefore a method-faithful reconstruction, not a
claim of bit-for-bit identity with unavailable original code.

Typical use
-----------
# 1) Clone JIT-Defect-Extended yourself:
#    git clone https://github.com/JIT-LSM/JIT-Defect-Extended.git
#
# 2) Build the commit cache and clone the 21 project repos if necessary:
#    python fuse_jit_jitdefect_extended.py prepare \
#       --dataset-root /path/JIT-Defect-Extended/dataset \
#       --repos-root /path/fusejit_repos --clone-repos \
#       --cache-dir /path/fusejit_cache
#
# 3) Run whole-dataset experiments over all 21 projects:
#    python fuse_jit_jitdefect_extended.py run \
#       --cache-dir /path/fusejit_cache \
#       --output-dir /path/results --runs 30

Python 3.10+ is recommended.
"""


# PROFESSIONAL STAGE-3 REVISION (2026-09)
# - One global 8:1:1 model over all 21 projects (not per-project training).
# - Training-only process normalization and cost-sensitive loss.
# - Rotation-invariant use of cached per-graph Node2Vec information.
# - Standard classification + effort-aware metrics.
# - Stage 1/2 persistent caches/checkpoints are fully reused; no re-extraction required.
#
# ACCURACY IMPROVEMENT PASS (2026-09, follow-up)
# - Expert-metrics view widened from 10 to the full 14 Kamei et al. (2013)
#   commit-level metrics by adding AGE/EXP/REXP/SEXP (developer experience and
#   recency), computed strictly from history up to the parent commit (no leakage).
#   This requires a one-time Stage-1 re-derivation; see _commit_records_schema_ok /
#   _refresh_process_metrics_if_possible for the automatic, non-destructive
#   upgrade path that avoids re-running the expensive Stage-2 AST/Node2Vec
#   extraction (only the process vectors are patched in place).
# - _load_or_validate_manifest now archives (never deletes) stale checkpoint
#   shards on a fingerprint change instead of raising, so this schema upgrade
#   (and any future one) applies automatically on the next run.
# - Added an auxiliary supervised-contrastive loss (Khosla et al., 2020) on the
#   fused embedding, matching the mechanism recent published fusion work on this
#   same 21-project benchmark reports large F1/AUC gains from over a
#   classification-loss-only fusion model. Disable via CONTRASTIVE_WEIGHT = 0.0.
# - Validation checkpoint/scheduler criterion changed from F1-only to a 50/50
#   F1+AUC blend, since both are reported target metrics.
# - Fixed a latent NumPy>=2.0 crash in effort_metrics() (np.trapz was removed).
# - Added a post-hoc cross-run bagging ensemble (+ an optional validation-gated
#   logistic-regression stacker) over the independently seeded runs sharing the
#   same fixed split -- see _write_ensemble_summary() / ensemble_summary.csv.
# - Widened the validation-tuned XGBoost booster's hyperparameter grid.
#
# ACCURACY IMPROVEMENT PASS 2 (2026-09, second follow-up)
# The above pass alone left F1/AUC essentially flat versus the pre-pass-1 baseline
# (observed: F1~0.43, AUC~0.84). Root cause: the syntax view is a Word2Vec model
# trained from scratch on only this dataset's ~22K training commits -- far weaker
# than the pretrained code-language-model embeddings that published work reaching
# the F1~0.52/AUC~0.91 target range on this exact benchmark actually relies on.
# - Added a fourth, independent view: a pretrained code-model (default
#   microsoft/codebert-base) [CLS] embedding of each commit's compact unified-diff
#   text (build_change_diff_text). Extracted ONCE for the whole dataset in a new,
#   independently resumable/checkpointed Stage 2.5 (extract_semantic_embeddings /
#   run_semantic_colab) that is deliberately decoupled from Stage 2: it reads
#   commit_records.pkl directly and needs no tree-sitter/Node2Vec rework, so it
#   never repeats that expensive pass when only the semantic model/config changes.
#   Disable via USE_SEMANTIC_EMBEDDINGS = False to fall back to the 3-view model.
# - FUSEJIT's fusion mechanism replaced: the old two-stage nested sigmoid gates
#   (which only combined a fixed pair of inputs at a time) are now a single
#   attention-weighted pooling over all 4 views (syntax/graph/process/semantic),
#   which scales cleanly to this 4th view and lets the model learn, per example,
#   how much to trust each one.
# - The semantic embedding is also fed to the XGBoost booster, via a train-only
#   PCA reduction to 32 components (raw pretrained-embedding dimensions are not
#   independently axis-aligned-split-friendly for a tree model the way engineered
#   features are, and 768 raw dims would badly outnumber the ~40 engineered ones).
# - Fixed a real cache bug found while adding this: the booster's persisted
#   val/test predictions were keyed only by commit-sha identity, with no schema
#   fingerprint, so a schema upgrade with the same split could have silently
#   reused stale-schema predictions. Now schema-fingerprinted like every other
#   persistent cache in this script.
# - Fixed a latent _deadline_reached() bug (`deadline and ...` treats an exact
#   0.0 deadline as "no deadline" via Python's falsy-zero short-circuit). Never
#   triggered by the real time.time()+hours deadlines this script computes, but
#   incorrect on that input regardless.
from __future__ import annotations

import argparse
import collections
import csv
import dataclasses
import hashlib
import json
import math
import multiprocessing as mp
import os
import pickle
import random
import re
import shutil
import subprocess
import sys
import time
from datetime import datetime
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, Iterable, Iterator, List, Mapping, Optional, Sequence, Set, Tuple

import numpy as np

# ----------------------------- constants -----------------------------------

PROJECT_REPOS: Dict[str, str] = {
    "ant-ivy": "https://github.com/apache/ant-ivy.git",
    "commons-bcel": "https://github.com/apache/commons-bcel.git",
    "commons-beanutils": "https://github.com/apache/commons-beanutils.git",
    "commons-codec": "https://github.com/apache/commons-codec.git",
    "commons-collections": "https://github.com/apache/commons-collections.git",
    "commons-compress": "https://github.com/apache/commons-compress.git",
    "commons-configuration": "https://github.com/apache/commons-configuration.git",
    "commons-dbcp": "https://github.com/apache/commons-dbcp.git",
    "commons-digester": "https://github.com/apache/commons-digester.git",
    "commons-io": "https://github.com/apache/commons-io.git",
    "commons-jcs": "https://github.com/apache/commons-jcs.git",
    "commons-lang": "https://github.com/apache/commons-lang.git",
    "commons-math": "https://github.com/apache/commons-math.git",
    "commons-net": "https://github.com/apache/commons-net.git",
    "commons-scxml": "https://github.com/apache/commons-scxml.git",
    "commons-validator": "https://github.com/apache/commons-validator.git",
    "commons-vfs": "https://github.com/apache/commons-vfs.git",
    "giraph": "https://github.com/apache/giraph.git",
    "gora": "https://github.com/apache/gora.git",
    "opennlp": "https://github.com/apache/opennlp.git",
    "parquet-mr": "https://github.com/apache/parquet-mr.git",
}

# The original 10 metrics plus the 4 experience/recency metrics from the
# established Kamei et al. (2013) commit-level feature set. AGE/EXP/REXP/SEXP
# consistently rank among the strongest predictors in JIT defect prediction
# literature, so adding them substantially strengthens the "expert metrics" view
# that both the neural process tower and the complementary booster consume.
PROCESS_FEATURES = [
    "NS", "ND", "NF", "Entropy", "FIX", "NUC", "LA", "LD", "LT", "NDEV",
    "AGE", "EXP", "REXP", "SEXP",
]

# Hidden size of the default pretrained encoder (CodeBERT-base) used for the
# semantic-embedding view; see build_change_diff_text/extract_semantic_embeddings.
SEMANTIC_DIM = 768

HEX_RE = re.compile(r"(?<![0-9a-fA-F])([0-9a-fA-F]{7,40})(?![0-9a-fA-F])")
FIX_RE = re.compile(r"\b(fix(?:e[ds])?|bug(?:s|fix)?|defect|patch|repair|issue)\b", re.I)
JAVA_IDENT_RE = re.compile(r"\b[A-Za-z_$][A-Za-z0-9_$]*\b")
LITERAL_RE = re.compile(
    r'("(?:\\.|[^"\\])*"|\'(?:\\.|[^\'\\])*\'|\b(?:0[xX][0-9a-fA-F]+|\d+(?:\.\d+)?(?:[eE][+-]?\d+)?[fFdDlL]?)\b)'
)

STATEMENT_TYPES = {
    "assert_statement", "break_statement", "continue_statement", "do_statement",
    "enhanced_for_statement", "expression_statement", "for_statement", "if_statement",
    "labeled_statement", "local_variable_declaration", "return_statement",
    "switch_expression", "switch_statement", "synchronized_statement", "throw_statement",
    "try_statement", "while_statement", "yield_statement", "explicit_constructor_invocation",
}
CONTROL_TYPES = {
    "if_statement", "for_statement", "enhanced_for_statement", "while_statement",
    "do_statement", "switch_statement", "switch_expression", "try_statement",
    "synchronized_statement",
}

# --------------------------- generic utilities ------------------------------

def stable_seed(*parts: Any, mod: int = 2**31 - 1) -> int:
    s = "|".join(map(str, parts)).encode("utf-8", "replace")
    return int(hashlib.sha256(s).hexdigest()[:16], 16) % mod


def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except Exception:
        pass


def run_cmd(cmd: Sequence[str], cwd: Optional[Path] = None, check: bool = True,
            text: bool = True, timeout: Optional[int] = None) -> subprocess.CompletedProcess:
    # Historical repositories can contain source/metadata bytes that are not valid
    # UTF-8.  Decode Git's textual output deterministically with replacement so a
    # legacy byte never aborts dataset reconstruction.  Binary callers still get
    # the original bytes unchanged.
    kwargs: Dict[str, Any] = {
        "cwd": str(cwd) if cwd else None,
        "stdout": subprocess.PIPE,
        "stderr": subprocess.PIPE,
        "text": text,
        "check": False,
        "timeout": timeout,
    }
    if text:
        kwargs.update(encoding="utf-8", errors="replace")
    p = subprocess.run(list(cmd), **kwargs)
    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}): {' '.join(map(str, cmd))}\n"
            f"STDOUT:\n{p.stdout}\nSTDERR:\n{p.stderr}"
        )
    return p


def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def save_json(obj: Any, path: Path) -> None:
    ensure_dir(path.parent)
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")


def load_json(path: Path) -> Any:
    return json.loads(path.read_text(encoding="utf-8", errors="replace"))


def safe_float(x: Any, default: float = 0.0) -> float:
    try:
        return float(x)
    except Exception:
        return default

# --------------------- JIT-Defect-Extended label discovery ------------------

@dataclass
class LabelEvidence:
    sha: str
    label: int
    source_paths: List[str] = field(default_factory=list)
    project_hint: Optional[str] = None
    timestamp_hint: Optional[str] = None


def _all_strings(obj: Any) -> Iterator[Tuple[Optional[str], str]]:
    if isinstance(obj, dict):
        for k, v in obj.items():
            if isinstance(v, str):
                yield str(k), v
            else:
                yield from _all_strings(v)
    elif isinstance(obj, list):
        for v in obj:
            yield from _all_strings(v)
    elif isinstance(obj, str):
        yield None, obj


def _extract_sha_candidates(text: str) -> List[str]:
    return [m.group(1).lower() for m in HEX_RE.finditer(text)]


def _json_record_candidates(obj: Any) -> List[Tuple[str, Optional[str], Optional[str]]]:
    """Return explicit commit-id candidates from arbitrary JSON.

    Important: do NOT fall back to scanning every string in the record. JIT datasets
    often contain related/fixing/parent commit identifiers in metadata; treating any
    SHA-looking string as the sample id can assign the same commit to both classes.
    """
    out: List[Tuple[str, Optional[str], Optional[str]]] = []
    records = obj if isinstance(obj, list) else [obj]
    if isinstance(obj, dict):
        # Some datasets use a dict keyed by commit id. Only promote a dict key when
        # the key itself is a SHA-looking identifier.
        records = []
        for k, v in obj.items():
            if isinstance(v, dict):
                vv = dict(v)
                if _extract_sha_candidates(str(k)):
                    vv.setdefault("_key", k)
                records.append(vv)
        if not records:
            records = [obj]

    for rec in records:
        if not isinstance(rec, dict):
            continue
        project = None
        ts = None
        for k in ("project", "project_name", "repo", "repository", "repo_name"):
            if k in rec and isinstance(rec[k], str):
                project = rec[k]
                break
        for k in ("author_date", "author_time", "timestamp", "date", "commit_date"):
            if k in rec and isinstance(rec[k], (str, int, float)):
                ts = str(rec[k])
                break

        # Only fields that can reasonably denote the sample / bug-inducing commit.
        prioritized_keys = (
            "commit_id", "commit", "commit_hash", "hash", "sha", "sha1",
            "bug_inducing_commit", "buggy_commit", "bic", "_key"
        )
        found: List[str] = []
        for k in prioritized_keys:
            if k in rec and rec[k] is not None:
                found.extend(_extract_sha_candidates(str(rec[k])))
        for sha in dict.fromkeys(found):
            out.append((sha, project, ts))
    return out


# Populated by discover_labels() and saved by cmd_prepare().
LAST_LABEL_CONFLICTS: List[Dict[str, Any]] = []


def _path_sha_candidates(path: Path) -> List[str]:
    """Extract SHA candidates from this path item only, not its whole relative path.

    The previous implementation scanned the full relative path for every descendant,
    so a SHA in one ancestor directory was repeatedly re-discovered from every file
    below it and unrelated SHA-looking path fragments could become labels.
    """
    names = [path.name]
    if path.is_file():
        names.append(path.stem)
    out: List[str] = []
    for name in names:
        out.extend(_extract_sha_candidates(name))
    return list(dict.fromkeys(out))


def _manifest_sha_candidates(line: str) -> List[str]:
    """Extract a commit id only from the first manifest field/token."""
    s = line.strip()
    if not s or s.startswith("#"):
        return []
    first = re.split(r"[\t,;\s]+", s, maxsplit=1)[0].strip('"\'')
    return _extract_sha_candidates(first)


def discover_labels(dataset_root: Path) -> Dict[str, LabelEvidence]:
    """Discover commit labels from JIT-Defect-Extended.

    `dataset/buggy` and `dataset/no_buggy` remain the authoritative class roots.
    Discovery is deliberately conservative: path evidence is taken from the current
    path item (not every ancestor), structured metadata uses explicit commit-id
    fields only, and manifest files use only their first field.

    If one SHA has evidence in both class roots, the conflict is retained for audit
    and collapsed to label=1 (buggy) for commit-level prediction. This follows the
    standard commit-level interpretation that a change is defective when there is
    positive defect-introducing evidence anywhere in that change, while avoiding a
    false-negative label. Every such case is written to label_conflicts.json by
    cmd_prepare(), so no conflict is hidden.
    """
    global LAST_LABEL_CONFLICTS
    LAST_LABEL_CONFLICTS = []

    roots = [(dataset_root / "buggy", 1), (dataset_root / "no_buggy", 0)]
    for root, _ in roots:
        if not root.exists():
            raise FileNotFoundError(
                f"Expected {root}. The JIT-Defect-Extended dataset should contain "
                "dataset/buggy and dataset/no_buggy."
            )

    # Keep class-specific evidence separately so mixed evidence can be audited rather
    # than aborting the whole reconstruction before Git validation.
    raw: Dict[str, Dict[int, LabelEvidence]] = collections.defaultdict(dict)

    def add(sha: str, label: int, src: Path, project: Optional[str] = None,
            ts: Optional[str] = None) -> None:
        sha = sha.lower()
        per_label = raw[sha]
        e = per_label.get(label)
        if e is None:
            e = LabelEvidence(sha=sha, label=label)
            per_label[label] = e
        sp = str(src)
        if sp not in e.source_paths:
            e.source_paths.append(sp)
        e.project_hint = e.project_hint or project
        e.timestamp_hint = e.timestamp_hint or ts

    for class_root, label in roots:
        # Path evidence: inspect each path item's own name only. This still supports
        # layouts such as <project>/<sha>/... and files whose basename contains SHA.
        for p in class_root.rglob("*"):
            for sha in _path_sha_candidates(p):
                add(sha, label, p)

        # Structured metadata evidence.
        for p in class_root.rglob("*"):
            if not p.is_file():
                continue
            suffix = p.suffix.lower()
            try:
                if suffix == ".json":
                    obj = load_json(p)
                    for sha, project, ts in _json_record_candidates(obj):
                        add(sha, label, p, project, ts)
                elif suffix == ".jsonl":
                    with p.open("r", encoding="utf-8", errors="replace") as f:
                        for line in f:
                            line = line.strip()
                            if not line:
                                continue
                            try:
                                obj = json.loads(line)
                            except Exception:
                                continue
                            for sha, project, ts in _json_record_candidates(obj):
                                add(sha, label, p, project, ts)
                elif suffix == ".csv":
                    with p.open("r", encoding="utf-8-sig", errors="replace", newline="") as f:
                        reader = csv.DictReader(f)
                        for row in reader:
                            project = next((row[k] for k in ("project", "project_name", "repo", "repository", "repo_name") if k in row and row[k]), None)
                            ts = next((row[k] for k in ("author_date", "author_time", "timestamp", "date", "commit_date") if k in row and row[k]), None)
                            found: List[str] = []
                            for k in ("commit_id", "commit", "commit_hash", "hash", "sha", "sha1", "bug_inducing_commit", "buggy_commit", "bic"):
                                if k in row and row[k]:
                                    found.extend(_extract_sha_candidates(str(row[k])))
                            for sha in dict.fromkeys(found):
                                add(sha, label, p, project, ts)
                elif suffix in {".txt", ".tsv", ".list"} and p.stat().st_size < 50_000_000:
                    with p.open("r", encoding="utf-8", errors="replace") as f:
                        for line in f:
                            for sha in _manifest_sha_candidates(line):
                                add(sha, label, p)
            except Exception as ex:
                print(f"[WARN] Could not parse metadata file {p}: {ex}", file=sys.stderr)

    if not raw:
        raise RuntimeError(
            "No commit SHA identifiers could be discovered in JIT-Defect-Extended. "
            "Use --labels-csv with columns sha,label[,project] if your checkout has "
            "been reorganized."
        )

    evid: Dict[str, LabelEvidence] = {}
    conflicts: List[Dict[str, Any]] = []
    for sha, per_label in raw.items():
        if len(per_label) == 1:
            evid[sha] = next(iter(per_label.values()))
            continue

        # Positive evidence dominates for commit-level prediction. Preserve all paths
        # so the conflict is fully auditable.
        pos = per_label.get(1)
        neg = per_label.get(0)
        chosen = pos if pos is not None else next(iter(per_label.values()))
        merged_paths: List[str] = []
        for e in per_label.values():
            for sp in e.source_paths:
                if sp not in merged_paths:
                    merged_paths.append(sp)
        chosen.source_paths = merged_paths
        if chosen.project_hint is None and neg is not None:
            chosen.project_hint = neg.project_hint
        if chosen.timestamp_hint is None and neg is not None:
            chosen.timestamp_hint = neg.timestamp_hint
        evid[sha] = chosen

        conflicts.append({
            "sha": sha,
            "resolved_label": int(chosen.label),
            "resolution": "buggy-dominates-for-commit-level-label",
            "buggy_sources": (pos.source_paths if pos is not None else []),
            "no_buggy_sources": (neg.source_paths if neg is not None else []),
            "buggy_project_hint": (pos.project_hint if pos is not None else None),
            "no_buggy_project_hint": (neg.project_hint if neg is not None else None),
        })

    LAST_LABEL_CONFLICTS = conflicts
    if conflicts:
        print(
            f"[labels] WARNING: {len(conflicts):,} SHA(s) had evidence in both "
            "buggy and no_buggy; resolved as buggy and retained for audit."
        )
        for c in conflicts[:10]:
            print(f"[labels] conflict {c['sha']} -> buggy")
        if len(conflicts) > 10:
            print(f"[labels] ... {len(conflicts) - 10:,} additional conflicts")

    return evid


def read_labels_csv(path: Path) -> Dict[str, LabelEvidence]:
    out: Dict[str, LabelEvidence] = {}
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        for row in csv.DictReader(f):
            sha = (row.get("sha") or row.get("commit") or row.get("commit_hash") or "").strip().lower()
            if not sha:
                continue
            label = int(row.get("label", row.get("buggy", "0")))
            out[sha] = LabelEvidence(
                sha=sha, label=label,
                source_paths=[str(path)],
                project_hint=(row.get("project") or None),
                timestamp_hint=(row.get("timestamp") or row.get("author_date") or None),
            )
    return out

# ------------------------------- Git layer ----------------------------------

class GitRepo:
    def __init__(self, name: str, path: Path):
        self.name = name
        self.path = path

    def git(self, *args: str, check: bool = True, text: bool = True) -> str:
        p = run_cmd(["git", *args], cwd=self.path, check=check, text=text)
        return p.stdout if text else p.stdout  # type: ignore[return-value]

    def resolve(self, sha: str) -> Optional[str]:
        p = run_cmd(["git", "rev-parse", "--verify", f"{sha}^{{commit}}"], cwd=self.path, check=False)
        return p.stdout.strip() if p.returncode == 0 else None

    def parent(self, sha: str) -> Optional[str]:
        p = run_cmd(["git", "rev-parse", f"{sha}^"], cwd=self.path, check=False)
        return p.stdout.strip() if p.returncode == 0 else None

    def show_text(self, rev: str, path: str) -> str:
        p = run_cmd(["git", "show", f"{rev}:{path}"], cwd=self.path, check=False)
        return p.stdout if p.returncode == 0 else ""

    def show_bytes(self, rev: str, path: str) -> bytes:
        p = subprocess.run(
            ["git", "show", f"{rev}:{path}"], cwd=str(self.path),
            stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=False,
        )
        return p.stdout if p.returncode == 0 else b""

    def metadata(self, sha: str) -> Dict[str, str]:
        fmt = "%H%x1f%P%x1f%aI%x1f%ae%x1f%s"
        out = self.git("show", "-s", f"--format={fmt}", sha).strip()
        parts = out.split("\x1f")
        if len(parts) < 5:
            raise RuntimeError(f"Unexpected git metadata for {sha}: {out!r}")
        return {
            "sha": parts[0], "parents": parts[1], "author_date": parts[2],
            "author_email": parts[3], "subject": parts[4],
        }

    def changed_java_files(self, parent: str, sha: str) -> List[Tuple[str, str]]:
        """Return list of (old_path,new_path); empty path denotes add/delete."""
        out = self.git("diff", "--name-status", "-M", parent, sha)
        pairs: List[Tuple[str, str]] = []
        for line in out.splitlines():
            cols = line.split("\t")
            if not cols:
                continue
            status = cols[0]
            if status.startswith("R") and len(cols) >= 3:
                old, new = cols[1], cols[2]
            elif status.startswith("A") and len(cols) >= 2:
                old, new = "", cols[1]
            elif status.startswith("D") and len(cols) >= 2:
                old, new = cols[1], ""
            elif len(cols) >= 2:
                old = new = cols[1]
            else:
                continue
            if (old.lower().endswith(".java") if old else False) or (new.lower().endswith(".java") if new else False):
                pairs.append((old, new))
        return pairs

    def numstat_java(self, parent: str, sha: str) -> Dict[str, Tuple[int, int]]:
        out = self.git("diff", "--numstat", "-M", parent, sha)
        d: Dict[str, Tuple[int, int]] = {}
        for line in out.splitlines():
            cols = line.split("\t")
            if len(cols) < 3:
                continue
            a, dele, path = cols[0], cols[1], cols[-1]
            if not path.lower().endswith(".java"):
                continue
            if a == "-" or dele == "-":
                continue
            d[path] = (int(a), int(dele))
        return d

    def prior_authors(self, parent: str, paths: Sequence[str]) -> Set[str]:
        if not paths:
            return set()
        p = run_cmd(["git", "log", parent, "--format=%ae", "--", *paths], cwd=self.path, check=False)
        return {x.strip().lower() for x in p.stdout.splitlines() if x.strip()}

    def prior_commits(self, parent: str, paths: Sequence[str]) -> Set[str]:
        if not paths:
            return set()
        p = run_cmd(["git", "log", parent, "--format=%H", "--", *paths], cwd=self.path, check=False)
        return {x.strip() for x in p.stdout.splitlines() if x.strip()}

    def author_history(self, parent: str, author_email: str, paths: Optional[Sequence[str]] = None) -> List[str]:
        """Author-dates (ISO 8601) of every commit by `author_email` reachable from
        `parent`, optionally restricted to `paths` (a subsystem directory list).

        The match pattern is anchored on the '<email>' delimiters that always
        surround the address in Git's author line and is matched case-insensitively,
        so it cannot be fooled by regex metacharacters in the address (e.g. '.', '+')
        or by one address being a substring of another (e.g. 'bob@x.com' vs
        'bob@x.com.au'). Filtering happens inside Git, so this stays cheap even for
        projects with long histories.
        """
        if not author_email:
            return []
        pattern = f"<{re.escape(author_email.lower())}>"
        args = ["git", "log", parent, "--extended-regexp", "--regexp-ignore-case",
                f"--author={pattern}", "--format=%aI"]
        if paths:
            args = args + ["--", *paths]
        p = run_cmd(args, cwd=self.path, check=False)
        if p.returncode != 0:
            return []
        return [x.strip() for x in p.stdout.splitlines() if x.strip()]

    def last_change_dates(self, parent: str, paths: Sequence[str]) -> Dict[str, str]:
        """Author-date (ISO 8601) of the most recent commit touching each of `paths`,
        as of `parent`. Renames are not followed, matching prior_authors/prior_commits.
        """
        if not paths:
            return {}
        p = run_cmd(["git", "log", parent, "--format=C\x1f%aI", "--name-only", "--", *paths],
                    cwd=self.path, check=False)
        if p.returncode != 0:
            return {}
        result: Dict[str, str] = {}
        cur_date: Optional[str] = None
        for line in p.stdout.splitlines():
            if line.startswith("C\x1f"):
                cur_date = line[2:].strip()
            elif line.strip() and cur_date is not None:
                fp = line.strip()
                if fp not in result:
                    result[fp] = cur_date
        return result


def clone_repositories(repos_root: Path, depth: Optional[int] = None) -> None:
    ensure_dir(repos_root)
    for name, url in PROJECT_REPOS.items():
        dest = repos_root / name
        if (dest / ".git").exists():
            print(f"[repo] {name}: already present")
            continue
        cmd = ["git", "clone"]
        # A full clone is safest because dataset commits are historical.
        if depth:
            cmd += ["--depth", str(depth)]
        cmd += [url, str(dest)]
        print(f"[repo] cloning {name}")
        run_cmd(cmd, check=True, timeout=None)


def load_repos(repos_root: Path) -> Dict[str, GitRepo]:
    repos: Dict[str, GitRepo] = {}
    for name in PROJECT_REPOS:
        p = repos_root / name
        if (p / ".git").exists():
            repos[name] = GitRepo(name, p)
    if not repos:
        raise FileNotFoundError(f"No Git repositories found under {repos_root}")
    return repos


def canonical_project_hint(hint: Optional[str]) -> Optional[str]:
    if not hint:
        return None
    s = hint.lower().replace(".git", "").strip().rstrip("/")
    s = s.split("/")[-1]
    aliases = {
        "ivy": "ant-ivy", "bcel": "commons-bcel", "beanutils": "commons-beanutils",
        "codec": "commons-codec", "collections": "commons-collections",
        "compress": "commons-compress", "configuration": "commons-configuration",
        "dbcp": "commons-dbcp", "digester": "commons-digester", "io": "commons-io",
        "jcs": "commons-jcs", "lang": "commons-lang", "math": "commons-math",
        "net": "commons-net", "scxml": "commons-scxml", "validator": "commons-validator",
        "vfs": "commons-vfs",
    }
    if s in PROJECT_REPOS:
        return s
    return aliases.get(s)


class ShaResolver:
    """Resolve many abbreviated/full SHAs with one `git rev-list --all` per repo."""
    def __init__(self, repos: Mapping[str, GitRepo]):
        self.repos = repos
        self.by7: Dict[str, List[Tuple[str, str]]] = collections.defaultdict(list)
        for name, repo in repos.items():
            print(f"[repo] indexing commits for {name}")
            out = repo.git("rev-list", "--all")
            for h in out.splitlines():
                h = h.strip().lower()
                if len(h) >= 7:
                    self.by7[h[:7]].append((name, h))

    def resolve(self, e: LabelEvidence) -> Tuple[Optional[str], Optional[str]]:
        sha = e.sha.lower()
        hint = canonical_project_hint(e.project_hint)
        candidates = [(n, h) for n, h in self.by7.get(sha[:7], []) if h.startswith(sha)]
        if hint:
            hinted = [(n, h) for n, h in candidates if n == hint]
            if len(hinted) == 1:
                return hinted[0]
        if len(candidates) == 1:
            return candidates[0]
        if len(candidates) > 1:
            raise RuntimeError(f"Commit prefix {sha} is ambiguous across repositories: {candidates[:8]}")
        return None, None


def map_sha_to_repo(e: LabelEvidence, repos: Mapping[str, GitRepo], resolver: Optional[ShaResolver] = None) -> Tuple[Optional[str], Optional[str]]:
    if resolver is not None:
        return resolver.resolve(e)
    # Fallback for callers resolving only a handful of commits.
    hint = canonical_project_hint(e.project_hint)
    if hint and hint in repos:
        full = repos[hint].resolve(e.sha)
        if full:
            return hint, full
    matches = []
    for name, repo in repos.items():
        full = repo.resolve(e.sha)
        if full:
            matches.append((name, full))
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        raise RuntimeError(f"Commit prefix {e.sha} is ambiguous across repositories: {matches}")
    return None, None

# --------------------------- commit reconstruction --------------------------

@dataclass
class FilePair:
    old_path: str
    new_path: str
    old_code: str
    new_code: str
    la: int = 0
    ld: int = 0

@dataclass
class CommitRecord:
    project: str
    sha: str
    label: int
    author_date: str
    author_email: str
    message: str
    parent: str
    file_pairs: List[FilePair]
    process: Dict[str, float]

    def to_dict(self) -> Dict[str, Any]:
        return dataclasses.asdict(self)

    @staticmethod
    def from_dict(d: Mapping[str, Any]) -> "CommitRecord":
        return CommitRecord(
            project=str(d["project"]), sha=str(d["sha"]), label=int(d["label"]),
            author_date=str(d["author_date"]), author_email=str(d.get("author_email", "")),
            message=str(d.get("message", "")), parent=str(d["parent"]),
            file_pairs=[FilePair(**fp) for fp in d["file_pairs"]],
            process={k: float(v) for k, v in d["process"].items()},
        )


def _path_for_stat(old: str, new: str) -> str:
    return new or old


def _loc(code: str) -> int:
    return sum(1 for line in code.splitlines() if line.strip())


def _parse_git_iso(s: Optional[str]) -> Optional[datetime]:
    if not s:
        return None
    try:
        return datetime.fromisoformat(s.strip())
    except Exception:
        return None


def process_metrics(repo: GitRepo, parent: str, sha: str, pairs: Sequence[Tuple[str, str]],
                    message: str, author_email: str, author_date: str) -> Dict[str, float]:
    paths = [_path_for_stat(a, b) for a, b in pairs]
    dirs = {str(Path(p).parent) for p in paths}
    subsystems = {Path(p).parts[0] if len(Path(p).parts) > 1 else "<root>" for p in paths}
    numstat = repo.numstat_java(parent, sha)

    churns: List[int] = []
    LA = LD = 0
    for p in paths:
        a, d = numstat.get(p, (0, 0))
        LA += a
        LD += d
        churns.append(a + d)
    total_churn = sum(churns)
    if total_churn > 0 and len(churns) > 1:
        ps = [c / total_churn for c in churns if c > 0]
        entropy = -sum(p * math.log(p, 2) for p in ps)
    else:
        entropy = 0.0

    LT = 0
    for old, new in pairs:
        p = old or new
        if old:
            LT += _loc(repo.show_text(parent, old))

    # Historical metrics are computed only from revisions chronologically prior
    # to the target change (the parent revision), preventing future leakage.
    hist_paths = [p for p in paths if p]
    ndev = len(repo.prior_authors(parent, hist_paths))
    nuc = len(repo.prior_commits(parent, hist_paths))

    # Experience/recency metrics (Kamei et al. 2013), also strictly bounded by
    # `parent` so nothing from the labeled commit itself or later leaks in.
    now = _parse_git_iso(author_date)
    author_hist = [d for d in (_parse_git_iso(x) for x in repo.author_history(parent, author_email)) if d is not None]
    exp = float(len(author_hist))
    if now is not None and author_hist:
        rexp = sum(1.0 / (max((now - d).days, 0) / 365.25 + 1.0) for d in author_hist)
    else:
        rexp = 0.0

    subsystem_dirs = sorted(s for s in subsystems if s != "<root>")
    sexp_hist = repo.author_history(parent, author_email, paths=subsystem_dirs) if subsystem_dirs else []
    sexp = float(len(sexp_hist))

    ages: List[float] = []
    if now is not None and hist_paths:
        last_dates = repo.last_change_dates(parent, hist_paths)
        for p in hist_paths:
            d = _parse_git_iso(last_dates.get(p))
            if d is not None:
                ages.append(float((now - d).days))
    age = (sum(ages) / len(ages)) if ages else 0.0

    return {
        "NS": float(len(subsystems)),
        "ND": float(len(dirs)),
        "NF": float(len(paths)),
        "Entropy": float(entropy),
        "FIX": float(bool(FIX_RE.search(message))),
        "NUC": float(nuc),
        "LA": float(LA),
        "LD": float(LD),
        "LT": float(LT),
        "NDEV": float(ndev),
        "AGE": float(age),
        "EXP": float(exp),
        "REXP": float(rexp),
        "SEXP": float(sexp),
    }


def build_commit_record(repo: GitRepo, full_sha: str, label: int,
                        max_files: int = 100, max_changed_loc: int = 10_000) -> Optional[CommitRecord]:
    parent = repo.parent(full_sha)
    if not parent:
        return None
    md = repo.metadata(full_sha)
    pairs = repo.changed_java_files(parent, full_sha)
    if not pairs:
        return None
    if len(pairs) > max_files:
        return None
    stat = repo.numstat_java(parent, full_sha)
    changed_loc = sum(a + d for a, d in stat.values())
    if changed_loc > max_changed_loc:
        return None

    fps: List[FilePair] = []
    for old, new in pairs:
        old_code = repo.show_text(parent, old) if old else ""
        new_code = repo.show_text(full_sha, new) if new else ""
        p = new or old
        la, ld = stat.get(p, (0, 0))
        # Non-functional/comment-only filtering is part of the benchmark
        # construction. Here we retain the benchmark sample and only skip empty
        # source pairs that cannot form either syntax or graph representations.
        if not old_code.strip() and not new_code.strip():
            continue
        fps.append(FilePair(old, new, old_code, new_code, la, ld))
    if not fps:
        return None

    proc = process_metrics(repo, parent, full_sha, pairs, md["subject"], md["author_email"], md["author_date"])
    return CommitRecord(
        project=repo.name, sha=full_sha, label=label,
        author_date=md["author_date"], author_email=md["author_email"],
        message=md["subject"], parent=parent, file_pairs=fps, process=proc,
    )

# -------------------------- Java AST representation -------------------------

@dataclass
class AstStmt:
    idx: int
    type: str
    text: str
    norm: str
    context: Tuple[str, ...]
    start: int
    end: int
    method: str
    defs: Tuple[str, ...] = ()
    uses: Tuple[str, ...] = ()
    calls: Tuple[str, ...] = ()


class JavaParser:
    def __init__(self):
        try:
            from tree_sitter import Language, Parser
            import tree_sitter_java
        except Exception as ex:
            raise RuntimeError(
                "Java parsing requires tree-sitter and tree-sitter-java. "
                "Install requirements_fuse_jit.txt."
            ) from ex
        self.Parser = Parser
        self.Language = Language
        self.tree_sitter_java = tree_sitter_java
        lang_obj = tree_sitter_java.language()
        try:
            language = Language(lang_obj)
        except Exception:
            language = lang_obj
        try:
            self.parser = Parser(language)
        except TypeError:
            self.parser = Parser()
            self.parser.set_language(language)

    @staticmethod
    def _text(src: bytes, node: Any) -> str:
        return src[node.start_byte:node.end_byte].decode("utf-8", "replace")

    @staticmethod
    def _walk(node: Any) -> Iterator[Any]:
        stack = [node]
        while stack:
            n = stack.pop()
            yield n
            children = list(n.children)
            stack.extend(reversed(children))

    def _method_name(self, src: bytes, method_node: Any) -> str:
        n = method_node.child_by_field_name("name")
        return self._text(src, n) if n is not None else "<method>"

    def _local_map(self, src: bytes, method_node: Any) -> Dict[str, str]:
        names: List[str] = []
        for n in self._walk(method_node):
            if n.type == "formal_parameter":
                x = n.child_by_field_name("name")
                if x is not None:
                    names.append(self._text(src, x))
            elif n.type == "variable_declarator":
                x = n.child_by_field_name("name")
                if x is not None:
                    names.append(self._text(src, x))
        out: Dict[str, str] = {}
        for name in names:
            if name not in out:
                out[name] = f"v{len(out)}"
        return out

    def _normalize(self, text: str, local_map: Mapping[str, str]) -> str:
        text = LITERAL_RE.sub("<LIT>", text)
        if local_map:
            text = JAVA_IDENT_RE.sub(lambda m: local_map.get(m.group(0), m.group(0)), text)
        text = re.sub(r"\s+", " ", text).strip()
        return text

    def _extract_defs_uses_calls(self, src: bytes, stmt_node: Any, local_map: Mapping[str, str],
                                 method: str) -> Tuple[Tuple[str, ...], Tuple[str, ...], Tuple[str, ...]]:
        defs: Set[str] = set()
        uses: Set[str] = set()
        calls: Set[str] = set()

        # Collect method invocations.
        for n in self._walk(stmt_node):
            if n.type == "method_invocation":
                name = n.child_by_field_name("name")
                if name is not None:
                    calls.add(self._text(src, name))

        # Definitions: declarators and assignment/update LHS.
        def_nodes: Set[Tuple[int, int]] = set()
        for n in self._walk(stmt_node):
            if n.type == "variable_declarator":
                name = n.child_by_field_name("name")
                if name is not None:
                    def_nodes.add((name.start_byte, name.end_byte))
                    defs.add(self._text(src, name))
            elif n.type == "assignment_expression":
                left = n.child_by_field_name("left")
                if left is not None:
                    ids = [x for x in self._walk(left) if x.type == "identifier"]
                    if ids:
                        x = ids[-1]
                        def_nodes.add((x.start_byte, x.end_byte))
                        defs.add(self._text(src, x))
            elif n.type == "update_expression":
                ids = [x for x in self._walk(n) if x.type == "identifier"]
                if ids:
                    x = ids[-1]
                    def_nodes.add((x.start_byte, x.end_byte))
                    defs.add(self._text(src, x))

        # Uses: all identifiers excluding method names/type identifiers and defs.
        for n in self._walk(stmt_node):
            if n.type != "identifier":
                continue
            key = (n.start_byte, n.end_byte)
            if key in def_nodes:
                continue
            name = self._text(src, n)
            parent_type = n.parent.type if n.parent is not None else ""
            if parent_type in {"method_invocation", "method_declaration", "class_declaration", "interface_declaration"}:
                # Keep receiver identifiers but skip direct invocation/declaration name.
                fld = None
                if n.parent is not None:
                    nm = n.parent.child_by_field_name("name")
                    if nm is not None and nm.start_byte == n.start_byte and nm.end_byte == n.end_byte:
                        continue
            uses.add(name)

        # Namespace local variables by method; fields/unresolved names remain global.
        def canon(name: str) -> str:
            return f"{method}::{local_map[name]}" if name in local_map else name
        return (
            tuple(sorted(canon(x) for x in defs)),
            tuple(sorted(canon(x) for x in uses)),
            tuple(sorted(calls)),
        )

    def statements(self, code: str) -> List[AstStmt]:
        if not code.strip():
            return []
        src = code.encode("utf-8", "replace")
        tree = self.parser.parse(src)
        root = tree.root_node
        methods: List[Any] = [n for n in self._walk(root) if n.type in {"method_declaration", "constructor_declaration"}]
        method_by_span: List[Tuple[int, int, Any, str, Dict[str, str]]] = []
        for m in methods:
            method_by_span.append((m.start_byte, m.end_byte, m, self._method_name(src, m), self._local_map(src, m)))

        stmts: List[AstStmt] = []
        for n in self._walk(root):
            if n.type not in STATEMENT_TYPES:
                continue
            enclosing = None
            for s, e, m, name, lmap in method_by_span:
                if s <= n.start_byte and n.end_byte <= e:
                    if enclosing is None or (e - s) < (enclosing[1] - enclosing[0]):
                        enclosing = (s, e, m, name, lmap)
            if enclosing:
                _, _, mnode, method_name, lmap = enclosing
            else:
                mnode, method_name, lmap = root, "<file>", {}
            ctx: List[str] = []
            cur = n.parent
            while cur is not None and cur != mnode.parent:
                ctx.append(cur.type)
                if cur == mnode:
                    break
                cur = cur.parent
            ctx.reverse()
            text = self._text(src, n)
            norm = self._normalize(text, lmap)
            defs, uses, calls = self._extract_defs_uses_calls(src, n, lmap, method_name)
            stmts.append(AstStmt(
                idx=len(stmts), type=n.type, text=text, norm=norm,
                context=tuple(ctx), start=n.start_byte, end=n.end_byte,
                method=method_name, defs=defs, uses=uses, calls=calls,
            ))
        stmts.sort(key=lambda x: (x.start, x.end))
        for i, s in enumerate(stmts):
            s.idx = i
        return stmts

# ------------------------ cross-version AST matching ------------------------

def context_similarity(a: Sequence[str], b: Sequence[str]) -> float:
    if not a and not b:
        return 1.0
    # Longest common suffix is useful for moved statements that retain their
    # immediate structural context.
    k = 0
    for x, y in zip(reversed(a), reversed(b)):
        if x != y:
            break
        k += 1
    return k / max(1, max(len(a), len(b)))


def match_statements(old: Sequence[AstStmt], new: Sequence[AstStmt]) -> Dict[int, int]:
    """
    Two-stage anchored matching corresponding to Section 4.1.2:
      1. AST/type + alpha-normalized text anchors, independent of position.
      2. Remaining candidates are resolved by <type, normalized code, context>.
    """
    old_by: Dict[Tuple[str, str], List[int]] = collections.defaultdict(list)
    new_by: Dict[Tuple[str, str], List[int]] = collections.defaultdict(list)
    for s in old:
        old_by[(s.type, s.norm)].append(s.idx)
    for s in new:
        new_by[(s.type, s.norm)].append(s.idx)
    m: Dict[int, int] = {}
    used_new: Set[int] = set()

    # Stage 1: unique anchors.
    for key, oids in old_by.items():
        nids = new_by.get(key, [])
        if len(oids) == 1 and len(nids) == 1:
            m[oids[0]] = nids[0]
            used_new.add(nids[0])

    # Stage 2: same type/norm; context and relative position break ties.
    for key, oids in old_by.items():
        candidates = [j for j in new_by.get(key, []) if j not in used_new]
        for oi in [x for x in oids if x not in m]:
            if not candidates:
                break
            best = max(
                candidates,
                key=lambda nj: (context_similarity(old[oi].context, new[nj].context),
                                -abs(old[oi].start - new[nj].start)),
            )
            m[oi] = best
            used_new.add(best)
            candidates.remove(best)
    return m


def lexical_tokens(text: str) -> List[str]:
    toks = re.findall(r"[A-Za-z_$][A-Za-z0-9_$]*|<LIT>|==|!=|<=|>=|&&|\|\||\S", text)
    return toks


def syntax_change_tokens(old: Sequence[AstStmt], new: Sequence[AstStmt], mapping: Mapping[int, int]) -> List[str]:
    matched_old = set(mapping)
    matched_new = set(mapping.values())
    changes: List[Tuple[int, List[str]]] = []
    for s in old:
        if s.idx not in matched_old:
            changes.append((s.start, [f"{s.type}:DEL", *lexical_tokens(s.norm)]))
    for s in new:
        if s.idx not in matched_new:
            changes.append((s.start, [f"{s.type}:ADD", *lexical_tokens(s.norm)]))
    changes.sort(key=lambda x: x[0])
    out: List[str] = []
    for _, toks in changes:
        out.extend(toks)
    return out

# ---------------------------- program graphs --------------------------------

@dataclass
class ProgramGraph:
    nodes: List[AstStmt]
    edges: Set[Tuple[int, int, str]]


def construct_program_graph(stmts: Sequence[AstStmt]) -> ProgramGraph:
    edges: Set[Tuple[int, int, str]] = set()
    if not stmts:
        return ProgramGraph(list(stmts), edges)

    # Control-flow: source-order successor within a method.
    by_method: Dict[str, List[AstStmt]] = collections.defaultdict(list)
    for s in stmts:
        by_method[s.method].append(s)
    for _, ss in by_method.items():
        ss = sorted(ss, key=lambda x: x.start)
        for a, b in zip(ss, ss[1:]):
            edges.add((a.idx, b.idx, "cf"))

    # Control-dependence approximation: nearest enclosing control statement.
    # Tree-sitter context names plus source nesting allow us to select the
    # nearest earlier control statement with a context prefix relation.
    controls = [s for s in stmts if s.type in CONTROL_TYPES]
    for s in stmts:
        cand = []
        for c in controls:
            if c.idx == s.idx or c.method != s.method or c.start > s.start:
                continue
            if len(c.context) <= len(s.context) and tuple(s.context[:len(c.context)]) == tuple(c.context):
                cand.append(c)
        if cand:
            c = max(cand, key=lambda x: x.start)
            edges.add((c.idx, s.idx, "cf"))

    # Data-flow: last-def -> use. Local identifiers are method-namespaced by
    # JavaParser; field/unresolved identifiers remain file-global and therefore
    # can create intra-file interprocedural def-use edges.
    last_def: Dict[str, int] = {}
    for s in sorted(stmts, key=lambda x: x.start):
        for u in s.uses:
            if u in last_def and last_def[u] != s.idx:
                edges.add((last_def[u], s.idx, "df"))
        for d in s.defs:
            last_def[d] = s.idx

    # Call edges to the first statement in a same-file target method.
    first_stmt: Dict[str, AstStmt] = {}
    for s in sorted(stmts, key=lambda x: x.start):
        first_stmt.setdefault(s.method, s)
    for s in stmts:
        for call in s.calls:
            t = first_stmt.get(call)
            if t is not None and t.idx != s.idx:
                edges.add((s.idx, t.idx, "call"))

    return ProgramGraph(list(stmts), edges)


@dataclass
class GraphChangeToken:
    kind: str           # node or edge
    sign: int           # +1 add / -1 delete
    relation: str       # node / cf / df / call
    node_type: str
    z_u: np.ndarray
    z_v: np.ndarray


def graph_changes(oldg: ProgramGraph, newg: ProgramGraph, mapping: Mapping[int, int],
                  old_emb: Mapping[int, np.ndarray], new_emb: Mapping[int, np.ndarray]) -> List[GraphChangeToken]:
    changes: List[Tuple[int, GraphChangeToken]] = []
    matched_old = set(mapping)
    matched_new = set(mapping.values())
    zdim = 100
    zero = np.zeros(zdim, dtype=np.float32)

    for s in oldg.nodes:
        if s.idx not in matched_old:
            changes.append((s.start, GraphChangeToken("node", -1, "node", s.type,
                                                     old_emb.get(s.idx, zero), zero)))
    for s in newg.nodes:
        if s.idx not in matched_new:
            changes.append((s.start, GraphChangeToken("node", +1, "node", s.type,
                                                     new_emb.get(s.idx, zero), zero)))

    mapped_old_edges: Set[Tuple[int, int, str]] = set()
    for u, v, r in oldg.edges:
        if u in mapping and v in mapping:
            mapped_old_edges.add((mapping[u], mapping[v], r))

    for u, v, r in oldg.edges:
        preserved = u in mapping and v in mapping and (mapping[u], mapping[v], r) in newg.edges
        if not preserved:
            pos = oldg.nodes[u].start if u < len(oldg.nodes) else 0
            changes.append((pos, GraphChangeToken(
                "edge", -1, r, oldg.nodes[u].type,
                old_emb.get(u, zero), old_emb.get(v, zero))))

    for u, v, r in newg.edges:
        if (u, v, r) not in mapped_old_edges:
            pos = newg.nodes[u].start if u < len(newg.nodes) else 0
            changes.append((pos, GraphChangeToken(
                "edge", +1, r, newg.nodes[u].type,
                new_emb.get(u, zero), new_emb.get(v, zero))))

    changes.sort(key=lambda x: x[0])
    return [c for _, c in changes]

# -------------------------- Node2Vec-style embedding ------------------------

def node2vec_embeddings(graph: ProgramGraph, dim: int = 100, window: int = 5,
                         walk_length: int = 20, num_walks: int = 10,
                         p: float = 1.0, q: float = 1.0, seed: int = 42) -> Dict[int, np.ndarray]:
    """
    Deterministic Node2Vec-style random-walk embedding for one program graph.

    The paper gives vector size=100 and context window=5 but does not report p,
    q, walk length, or walk count. Defaults p=q=1, walk_length=20, num_walks=10
    are therefore exposed as CLI configuration rather than hidden constants.
    """
    if not graph.nodes:
        return {}
    try:
        from gensim.models import Word2Vec
    except Exception as ex:
        raise RuntimeError("Node2Vec embedding requires gensim.") from ex

    rng = random.Random(seed)
    adj: Dict[int, Set[int]] = {s.idx: set() for s in graph.nodes}
    for u, v, _ in graph.edges:
        adj.setdefault(u, set()).add(v)
        adj.setdefault(v, set()).add(u)

    def weighted_next(prev: Optional[int], cur: int) -> Optional[int]:
        nbrs = list(adj.get(cur, ()))
        if not nbrs:
            return None
        if prev is None or (p == 1.0 and q == 1.0):
            return rng.choice(nbrs)
        weights = []
        prev_n = adj.get(prev, set())
        for x in nbrs:
            if x == prev:
                w = 1.0 / p
            elif x in prev_n:
                w = 1.0
            else:
                w = 1.0 / q
            weights.append(w)
        total = sum(weights)
        r = rng.random() * total
        acc = 0.0
        for x, w in zip(nbrs, weights):
            acc += w
            if acc >= r:
                return x
        return nbrs[-1]

    walks: List[List[str]] = []
    nodes = list(adj)
    for _ in range(num_walks):
        rng.shuffle(nodes)
        for start in nodes:
            walk = [start]
            prev = None
            cur = start
            for _step in range(max(0, walk_length - 1)):
                nxt = weighted_next(prev, cur)
                if nxt is None:
                    break
                walk.append(nxt)
                prev, cur = cur, nxt
            walks.append([str(x) for x in walk])

    model = Word2Vec(
        sentences=walks, vector_size=dim, window=window,
        min_count=1, sg=1, workers=1, epochs=5, seed=seed,
        batch_words=50,
    )
    return {i: np.asarray(model.wv[str(i)], dtype=np.float32) for i in adj}

# -------------------------- feature cache creation --------------------------

@dataclass
class FileFeatures:
    syntax_tokens: List[str]
    graph_tokens: List[GraphChangeToken]

@dataclass
class CommitFeatures:
    project: str
    sha: str
    label: int
    author_date: str
    process: np.ndarray  # (len(PROCESS_FEATURES),)
    effort: float
    files: List[FileFeatures]
    # Pretrained-code-model embedding of the commit's diff (see extract_semantic_embeddings).
    # Optional/defaulted so CommitFeatures pickled before this field existed still unpickle
    # cleanly; _commit_features_schema_ok() below detects the missing field and triggers the
    # (cheap, GPU-batched, no AST/Node2Vec rework needed) Stage-2.5 backfill.
    semantic_emb: Optional[np.ndarray] = None  # (SEMANTIC_DIM,) or None if not yet extracted


def featurize_record(record: CommitRecord, parser: JavaParser, cfg: "FeatureConfig") -> CommitFeatures:
    ffs: List[FileFeatures] = []
    for fp in record.file_pairs:
        old_st = parser.statements(fp.old_code)
        new_st = parser.statements(fp.new_code)
        mapping = match_statements(old_st, new_st)
        syn = syntax_change_tokens(old_st, new_st, mapping)
        oldg = construct_program_graph(old_st)
        newg = construct_program_graph(new_st)
        seed0 = stable_seed(record.project, record.sha, fp.old_path, fp.new_path)
        olde = node2vec_embeddings(
            oldg, dim=cfg.graph_dim, window=cfg.graph_window,
            walk_length=cfg.walk_length, num_walks=cfg.num_walks,
            p=cfg.node2vec_p, q=cfg.node2vec_q, seed=seed0,
        )
        newe = node2vec_embeddings(
            newg, dim=cfg.graph_dim, window=cfg.graph_window,
            walk_length=cfg.walk_length, num_walks=cfg.num_walks,
            p=cfg.node2vec_p, q=cfg.node2vec_q, seed=seed0 + 1,
        )
        gtok = graph_changes(oldg, newg, mapping, olde, newe)
        ffs.append(FileFeatures(syntax_tokens=syn, graph_tokens=gtok))
    proc = np.asarray([record.process[k] for k in PROCESS_FEATURES], dtype=np.float32)
    effort = max(1.0, float(record.process["LA"] + record.process["LD"]))
    return CommitFeatures(record.project, record.sha, record.label, record.author_date, proc, effort, ffs)


@dataclass
class FeatureConfig:
    syntax_dim: int = 100
    syntax_window: int = 10
    seq_len: int = 50
    graph_dim: int = 100
    graph_window: int = 5
    walk_length: int = 20
    num_walks: int = 10
    node2vec_p: float = 1.0
    node2vec_q: float = 1.0
    semantic_dim: int = SEMANTIC_DIM

# ---------------------- pretrained semantic embedding text -------------------
#
# The from-scratch Word2Vec syntax tower is trained on only this dataset's ~22K
# training commits, which is a far weaker semantic signal than a language model
# pretrained on millions of code files. Published JIT-DP fusion work that reaches
# the F1/AUC levels this reconstruction targets (JIT-Fine/JIT-Smart/JIT-Coka-style
# approaches) uses exactly that: a pretrained code-model embedding of the change,
# fused with the same kind of expert/process metrics this script already computes.
# build_change_diff_text() -> extract_semantic_embeddings() (near cmd_featurize)
# adds that as a fourth, independent view -- on top of, not instead of, the
# existing syntax/graph/process views.


def build_change_diff_text(record: "CommitRecord", max_chars: int = 6000) -> str:
    """Compact unified-diff text of a commit's changes, sized for a pretrained
    code-model tokenizer. A 1-line context window keeps the text concentrated on
    the actually-changed regions (not whole files), so even multi-file commits
    usually fit well within a 256/512-token budget.
    """
    import difflib
    parts: List[str] = []
    for fp in record.file_pairs:
        old_lines = fp.old_code.splitlines()
        new_lines = fp.new_code.splitlines()
        path = fp.new_path or fp.old_path
        hunk_lines: List[str] = [f"--- {path}"]
        for line in difflib.unified_diff(old_lines, new_lines, n=1, lineterm=""):
            # Skip difflib's own file-header lines ('---'/'+++'); the path line
            # above already identifies the file and every remaining char is spent
            # on '@@' hunk markers and actual +/- changed/context lines instead.
            if line.startswith("---") or line.startswith("+++"):
                continue
            hunk_lines.append(line)
        parts.append("\n".join(hunk_lines))
    text = "\n".join(parts).strip()
    return text[:max_chars]

# -------------------------- Word2Vec + tensors -------------------------------

def train_syntax_word2vec(train_features: Sequence[CommitFeatures], cfg: FeatureConfig, seed: int):
    try:
        from gensim.models import Word2Vec
    except Exception as ex:
        raise RuntimeError("Syntax embedding requires gensim.") from ex
    sentences: List[List[str]] = []
    for cf in train_features:
        for ff in cf.files:
            if ff.syntax_tokens:
                sentences.append(ff.syntax_tokens)
    if not sentences:
        sentences = [["<EMPTY>"]]
    return Word2Vec(
        sentences=sentences, vector_size=cfg.syntax_dim, window=cfg.syntax_window,
        min_count=1, sg=1, workers=1, epochs=10, seed=seed,
        batch_words=4,
    )


def syntax_map(tokens: Sequence[str], w2v: Any, cfg: FeatureConfig) -> np.ndarray:
    arr = np.zeros((cfg.seq_len, cfg.syntax_dim), dtype=np.float32)
    for i, tok in enumerate(tokens[:cfg.seq_len]):
        if tok in w2v.wv:
            arr[i] = w2v.wv[tok]
        else:
            # Dedicated deterministic UNK vector; training-only model remains frozen.
            rng = np.random.default_rng(stable_seed("UNK", tok))
            arr[i] = rng.normal(0, 0.01, cfg.syntax_dim).astype(np.float32)
    return arr


def commit_syntax_tensor(cf: CommitFeatures, w2v: Any, cfg: FeatureConfig) -> np.ndarray:
    maps = [syntax_map(ff.syntax_tokens, w2v, cfg) for ff in cf.files]
    if not maps:
        return np.zeros((cfg.seq_len, cfg.syntax_dim), dtype=np.float32)
    return np.max(np.stack(maps, axis=0), axis=0)

# ------------------------------ PyTorch model -------------------------------

def _torch_modules():
    try:
        import torch
        import torch.nn as nn
        import torch.nn.functional as F
        from torch.utils.data import Dataset, DataLoader
        return torch, nn, F, Dataset, DataLoader
    except Exception as ex:
        raise RuntimeError("Training requires PyTorch.") from ex


class GraphVocab:
    def __init__(self, features: Sequence[CommitFeatures]):
        types = {"<UNK>"}
        rels = {"node", "cf", "df", "call", "<UNK>"}
        for cf in features:
            for ff in cf.files:
                for t in ff.graph_tokens:
                    types.add(t.node_type)
                    rels.add(t.relation)
        self.type_to_id = {x: i for i, x in enumerate(sorted(types))}
        self.rel_to_id = {x: i for i, x in enumerate(sorted(rels))}

    def tid(self, s: str) -> int:
        return self.type_to_id.get(s, self.type_to_id["<UNK>"])

    def rid(self, s: str) -> int:
        return self.rel_to_id.get(s, self.rel_to_id["<UNK>"])


def graph_file_arrays(ff: FileFeatures, vocab: GraphVocab, cfg: FeatureConfig) -> Dict[str, np.ndarray]:
    L, D = cfg.seq_len, cfg.graph_dim
    zu = np.zeros((L, D), dtype=np.float32)
    zv = np.zeros((L, D), dtype=np.float32)
    typ = np.zeros(L, dtype=np.int64)
    rel = np.zeros(L, dtype=np.int64)
    sign = np.zeros(L, dtype=np.int64)
    mask = np.zeros(L, dtype=np.float32)
    for i, t in enumerate(ff.graph_tokens[:L]):
        zu[i] = t.z_u[:D]
        zv[i] = t.z_v[:D]
        typ[i] = vocab.tid(t.node_type)
        rel[i] = vocab.rid(t.relation)
        sign[i] = 1 if t.sign > 0 else 0
        mask[i] = 1.0
    return {"zu": zu, "zv": zv, "type": typ, "rel": rel, "sign": sign, "mask": mask}


def _graph_token_invariants(zu: np.ndarray, zv: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """Rotation-invariant summaries of independently trained per-graph Node2Vec vectors.

    A separate Node2Vec model is fitted for each program graph during Stage 2. Raw
    coordinate axes are therefore arbitrary across graphs/commits. Norms, cosine
    similarity and Euclidean distance are invariant to a common orthogonal rotation
    and can be compared safely by the downstream global classifier.
    """
    un = np.linalg.norm(zu, axis=1)
    vn = np.linalg.norm(zv, axis=1)
    dot = np.sum(zu * zv, axis=1)
    denom = np.maximum(un * vn, 1e-12)
    cos = np.where((un > 0) & (vn > 0), dot / denom, 0.0)
    dist = np.linalg.norm(zu - zv, axis=1)
    normdiff = np.abs(un - vn)
    has_v = (vn > 0).astype(np.float32)
    inv = np.stack([
        np.log1p(un), np.log1p(vn), cos,
        np.log1p(dist), np.log1p(normdiff), has_v,
    ], axis=1).astype(np.float32)
    return inv * mask[:, None].astype(np.float32)


def commit_graph_arrays(cf: CommitFeatures, vocab: GraphVocab, cfg: FeatureConfig) -> Dict[str, np.ndarray]:
    """Build a coherent fixed-length graph-change map for a commit.

    Stage-2 Node2Vec embeddings are independently fitted per file graph, so their raw
    coordinate axes cannot be element-wise mixed across files. We retain the same
    graph changes and Node2Vec information but convert each token to six invariant
    structural quantities, then select the strongest file token at each sequence
    position. Type/relation/sign are taken from that same token. No Stage-2 rerun is
    required and no test information is used here.
    """
    L = cfg.seq_len
    if not cf.files:
        zi = np.zeros(L, np.int64)
        zm = np.zeros(L, np.float32)
        return {
            "inv": np.zeros((L, 6), np.float32),
            "type": zi.copy(), "rel": zi.copy(), "sign": zi.copy(), "mask": zm,
        }

    arrs = [graph_file_arrays(ff, vocab, cfg) for ff in cf.files]
    invs = [_graph_token_invariants(a["zu"], a["zv"], a["mask"]) for a in arrs]
    inv_stack = np.stack(invs, axis=0)  # [files, L, 6]
    # Strength is based only on invariant endpoint magnitude/distance.
    score = inv_stack[:, :, 0] + inv_stack[:, :, 1] + inv_stack[:, :, 3]
    score = np.where(np.stack([a["mask"] for a in arrs], axis=0) > 0, score, -1.0)
    winner = np.argmax(score, axis=0)
    pos = np.arange(L)
    return {
        "inv": inv_stack[winner, pos],
        "type": np.stack([a["type"] for a in arrs], axis=0)[winner, pos],
        "rel": np.stack([a["rel"] for a in arrs], axis=0)[winner, pos],
        "sign": np.stack([a["sign"] for a in arrs], axis=0)[winner, pos],
        "mask": np.max(np.stack([a["mask"] for a in arrs], axis=0), axis=0),
    }


class NumpyDataset:
    def __init__(self, items: Sequence[Dict[str, Any]]):
        self.items = list(items)
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        return self.items[i]


def collate_batch(batch: Sequence[Dict[str, Any]]) -> Dict[str, Any]:
    torch, _, _, _, _ = _torch_modules()
    def stack(name, dtype=None):
        x = np.stack([b[name] for b in batch], axis=0)
        return torch.as_tensor(x, dtype=dtype)
    return {
        "syntax": stack("syntax", torch.float32),
        "g_inv": stack("g_inv", torch.float32),
        "g_type": stack("g_type", torch.long),
        "g_rel": stack("g_rel", torch.long),
        "g_sign": stack("g_sign", torch.long),
        "g_mask": stack("g_mask", torch.float32),
        "process": stack("process", torch.float32),
        "semantic": stack("semantic", torch.float32),
        "label": stack("label", torch.long),
        "effort": stack("effort", torch.float32),
        "sha": [b["sha"] for b in batch],
    }


def build_model(vocab: GraphVocab, cfg: FeatureConfig, dropout_embedding=0.3, dropout_branch=0.4):
    torch, nn, F, _, _ = _torch_modules()

    class SyntaxTower(nn.Module):
        def __init__(self):
            super().__init__()
            self.edrop = nn.Dropout(dropout_embedding)
            self.conv = nn.Conv1d(cfg.syntax_dim, 128, kernel_size=5, padding=0)
            self.norm = nn.LayerNorm(128)
            self.drop = nn.Dropout(dropout_branch)
            self.fc = nn.Linear(128, 64)
        def forward(self, x):
            x = self.edrop(x).transpose(1, 2)
            x = F.gelu(self.conv(x))
            x = torch.amax(x, dim=2)
            x = self.norm(x)
            return F.gelu(self.fc(self.drop(x)))

    class GraphTower(nn.Module):
        def __init__(self):
            super().__init__()
            self.type_emb = nn.Embedding(len(vocab.type_to_id), 16)
            self.rel_emb = nn.Embedding(len(vocab.rel_to_id), 8)
            self.sign_emb = nn.Embedding(2, 4)
            # 6 invariant Node2Vec-derived structural quantities + categorical change data.
            self.proj = nn.Linear(6 + 16 + 8 + 4, cfg.graph_dim)
            self.edrop = nn.Dropout(dropout_embedding)
            self.conv = nn.Conv1d(cfg.graph_dim, 128, kernel_size=5, padding=0)
            self.norm = nn.LayerNorm(128)
            self.drop = nn.Dropout(dropout_branch)
            self.fc = nn.Linear(128, 64)
        def forward(self, inv, typ, rel, sign, mask):
            x = torch.cat([inv, self.type_emb(typ), self.rel_emb(rel), self.sign_emb(sign)], dim=-1)
            x = F.gelu(self.proj(x)) * mask.unsqueeze(-1)
            x = self.edrop(x).transpose(1, 2)
            x = F.gelu(self.conv(x))
            # Masked inputs are zeros; max pooling preserves the original fixed-length design.
            x = torch.amax(x, dim=2)
            x = self.norm(x)
            return F.gelu(self.fc(self.drop(x)))

    class ProcessTower(nn.Module):
        def __init__(self):
            super().__init__()
            self.conv = nn.Conv1d(1, 128, kernel_size=9, padding=8)
            self.norm = nn.LayerNorm(128)
            self.drop = nn.Dropout(dropout_branch)
            self.fc64 = nn.Linear(128, 64)
            self.fc32 = nn.Linear(64, 32)
        def forward(self, x):
            x = x.unsqueeze(1)
            x = F.gelu(self.conv(x))
            x = torch.amax(x, dim=2)
            x = self.norm(x)
            x = F.gelu(self.fc64(self.drop(x)))
            return F.gelu(self.fc32(x))

    class SemanticTower(nn.Module):
        """Consumes the pretrained code-model [CLS] embedding of the commit's diff
        (see build_change_diff_text/extract_semantic_embeddings). Unlike the other
        three towers, its input already carries rich pretrained code understanding,
        so this is deliberately a small MLP rather than another conv stack."""
        def __init__(self):
            super().__init__()
            self.fc1 = nn.Linear(cfg.semantic_dim, 128)
            self.norm = nn.LayerNorm(128)
            self.drop = nn.Dropout(dropout_branch)
            self.fc2 = nn.Linear(128, 64)
        def forward(self, x):
            x = F.gelu(self.fc1(x))
            x = self.norm(x)
            return F.gelu(self.fc2(self.drop(x)))

    class FUSEJIT(nn.Module):
        """Four views (syntax/graph/process/semantic) fused by soft attention pooling:
        each view is projected to a common dimension, scored by a shared attention
        head, and combined by the resulting softmax weights. This replaces the
        earlier two-stage nested sigmoid-gate fusion (which only scaled to a fixed
        pair of inputs at a time) with a design that extends to any number of views
        and lets the model learn, per example, how much to trust each one -- e.g.
        leaning on the pretrained semantic view when the AST/graph views are thin."""
        def __init__(self):
            super().__init__()
            self.syn = SyntaxTower()
            self.graph = GraphTower()
            self.proc = ProcessTower()
            self.sem = SemanticTower()
            d = 32
            self.syn_proj = nn.Linear(64, d)
            self.graph_proj = nn.Linear(64, d)
            self.proc_proj = nn.Linear(32, d)
            self.sem_proj = nn.Linear(64, d)
            self.attn = nn.Linear(d, 1)
            self.out = nn.Linear(d, 2)
        def forward(self, b, return_embedding: bool = False):
            hs = self.syn(b["syntax"])
            hg = self.graph(b["g_inv"], b["g_type"], b["g_rel"], b["g_sign"], b["g_mask"])
            hp = self.proc(b["process"])
            hc = self.sem(b["semantic"])
            views = torch.stack([
                F.gelu(self.syn_proj(hs)),
                F.gelu(self.graph_proj(hg)),
                F.gelu(self.proc_proj(hp)),
                F.gelu(self.sem_proj(hc)),
            ], dim=1)  # [B, 4, d]
            weights = torch.softmax(self.attn(views).squeeze(-1), dim=1).unsqueeze(-1)  # [B, 4, 1]
            fm = (weights * views).sum(dim=1)  # [B, d]
            logits = self.out(fm)
            if return_embedding:
                return logits, fm
            return logits

    return FUSEJIT()


def supervised_contrastive_loss(torch_mod: Any, z: Any, labels: Any, temperature: float = 0.1) -> Any:
    """Supervised contrastive loss (Khosla et al., 2020) over the fused embedding `fm`.

    Sharpening class separation in the fused representation with a contrastive
    objective, on top of the same semantic (syntax/graph) + expert (process) feature
    fusion this model already performs, is the mechanism recent published work on
    this exact 21-project JIT-defect benchmark (contrastive learning + feature
    fusion) reports large F1/AUC gains from over a plain classification loss alone.

    Anchors with no other same-label example in the current minibatch contribute
    zero (not NaN): with ~8% positives, a batch can occasionally hold only one
    member of a class. A large-but-finite sentinel (not -inf) is used to mask
    self-similarity so `0 * masked_value` never produces NaN in the weighted sum.
    """
    torch = torch_mod
    n = int(z.shape[0])
    if n < 2:
        return z.sum() * 0.0
    zn = z / z.norm(dim=1, keepdim=True).clamp_min(1e-12)
    sim = torch.matmul(zn, zn.t()) / temperature
    self_mask = torch.eye(n, dtype=torch.bool, device=z.device)
    sim = sim.masked_fill(self_mask, -1e9)
    sim = sim - sim.max(dim=1, keepdim=True).values.detach()
    exp_sim = torch.exp(sim)
    denom = exp_sim.sum(dim=1, keepdim=True).clamp_min(1e-12)
    log_prob = sim - torch.log(denom)
    pos_mask = ((labels.unsqueeze(0) == labels.unsqueeze(1)) & (~self_mask)).float()
    pos_count = pos_mask.sum(dim=1)
    mean_log_prob_pos = (pos_mask * log_prob).sum(dim=1) / pos_count.clamp_min(1.0)
    valid = (pos_count > 0).float()
    if float(valid.sum()) < 1.0:
        return z.sum() * 0.0
    return -(mean_log_prob_pos * valid).sum() / valid.sum()


# ---------------------------- experiment logic ------------------------------

METRIC_NAMES = (
    "Prec", "Recall", "F1", "AUC", "MCC", "Accuracy", "Balanced_Accuracy",
    "PofB20", "R@20%E", "E@20%R", "Popt", "Top10", "Top5", "IFA",
)


def global_random_split(items: Sequence[CommitFeatures], seed: int = 42,
                        stratify: bool = False) -> Tuple[List[CommitFeatures], List[CommitFeatures], List[CommitFeatures]]:
    """Whole-dataset random 8:1:1 split.

    JIT-CF reports an 8:1:1 random train/validation/test partition of the complete
    21-project JIT-Defects4J benchmark. JIT-Defect-Extended follows the same benchmark
    lineage. The default here is the literal whole-dataset random protocol; optional
    label stratification is exposed but OFF by default for protocol fidelity.
    """
    xs = list(items)
    if not xs:
        return [], [], []
    rng = np.random.default_rng(seed)
    if not stratify:
        idx = np.arange(len(xs))
        rng.shuffle(idx)
        n_train = int(math.floor(0.80 * len(idx)))
        n_val = int(math.floor(0.10 * len(idx)))
        train_idx = idx[:n_train]
        val_idx = idx[n_train:n_train + n_val]
        test_idx = idx[n_train + n_val:]
    else:
        train_idx, val_idx, test_idx = [], [], []
        for label in (0, 1):
            ii = np.asarray([i for i, x in enumerate(xs) if x.label == label], dtype=int)
            rng.shuffle(ii)
            n_train = int(math.floor(0.80 * len(ii)))
            n_val = int(math.floor(0.10 * len(ii)))
            train_idx.extend(ii[:n_train].tolist())
            val_idx.extend(ii[n_train:n_train + n_val].tolist())
            test_idx.extend(ii[n_train + n_val:].tolist())
        rng.shuffle(train_idx); rng.shuffle(val_idx); rng.shuffle(test_idx)
    return ([xs[int(i)] for i in train_idx],
            [xs[int(i)] for i in val_idx],
            [xs[int(i)] for i in test_idx])


# Only the binary FIX indicator is left untransformed; every count/duration-like
# metric (including the new AGE/EXP/REXP/SEXP experience metrics) is log1p'd.
# Derived by name rather than position so PROCESS_FEATURES can be extended safely.
_PROCESS_NOLOG_FEATURES = {"FIX"}
_PROCESS_LOG_MASK = np.asarray([0 if f in _PROCESS_NOLOG_FEATURES else 1 for f in PROCESS_FEATURES], dtype=bool)


def _process_pretransform(x: np.ndarray) -> np.ndarray:
    y = np.asarray(x, dtype=np.float32).copy()
    y[_PROCESS_LOG_MASK] = np.log1p(np.maximum(y[_PROCESS_LOG_MASK], 0.0))
    return y


def fit_process_scaler(train_features: Sequence[CommitFeatures]) -> Tuple[np.ndarray, np.ndarray]:
    X = np.stack([_process_pretransform(cf.process) for cf in train_features], axis=0)
    mean = X.mean(axis=0).astype(np.float32)
    std = X.std(axis=0, ddof=0).astype(np.float32)
    std = np.where(std < 1e-6, 1.0, std).astype(np.float32)
    return mean, std


def transform_process(x: np.ndarray, stats: Tuple[np.ndarray, np.ndarray]) -> np.ndarray:
    mean, std = stats
    y = (_process_pretransform(x) - mean) / std
    return np.clip(y, -5.0, 5.0).astype(np.float32)


def _semantic_vector(cf: CommitFeatures, dim: int) -> np.ndarray:
    """L2-normalized pretrained embedding, or zeros if not (yet) extracted.

    A per-sample unit-norm rescale is enough preprocessing for a pretrained
    transformer's already-LayerNorm'd hidden state -- no train-fitted mean/std is
    needed, so this has no leakage surface (unlike fit_process_scaler).
    """
    v = cf.semantic_emb
    if v is None:
        return np.zeros(dim, dtype=np.float32)
    v = np.asarray(v, dtype=np.float32)
    n = float(np.linalg.norm(v))
    return (v / n).astype(np.float32) if n > 1e-12 else v


def prepare_numpy_items(features: Sequence[CommitFeatures], w2v: Any, vocab: GraphVocab,
                        cfg: FeatureConfig, process_stats: Tuple[np.ndarray, np.ndarray]) -> List[Dict[str, Any]]:
    out = []
    for cf in features:
        g = commit_graph_arrays(cf, vocab, cfg)
        out.append({
            "syntax": commit_syntax_tensor(cf, w2v, cfg),
            "g_inv": g["inv"], "g_type": g["type"], "g_rel": g["rel"],
            "g_sign": g["sign"], "g_mask": g["mask"],
            "process": transform_process(cf.process, process_stats),
            "semantic": _semantic_vector(cf, cfg.semantic_dim),
            "label": np.asarray(cf.label, dtype=np.int64),
            "effort": np.asarray(cf.effort, dtype=np.float32),
            "sha": cf.sha,
        })
    return out


def _confusion(y: np.ndarray, pred: np.ndarray) -> Tuple[int, int, int, int]:
    y = np.asarray(y, int); pred = np.asarray(pred, int)
    tp = int(np.sum((y == 1) & (pred == 1)))
    fp = int(np.sum((y == 0) & (pred == 1)))
    fn = int(np.sum((y == 1) & (pred == 0)))
    tn = int(np.sum((y == 0) & (pred == 0)))
    return tp, fp, fn, tn


def binary_f1(y: np.ndarray, pred: np.ndarray) -> float:
    tp, fp, fn, _ = _confusion(y, pred)
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    return 2.0 * p * r / (p + r) if p + r else 0.0


def binary_auc(y: np.ndarray, prob: np.ndarray) -> float:
    """ROC AUC via average ranks; handles tied probabilities without sklearn."""
    y = np.asarray(y, int); prob = np.asarray(prob, float)
    n_pos = int(np.sum(y == 1)); n_neg = int(np.sum(y == 0))
    if n_pos == 0 or n_neg == 0:
        return 0.0
    order = np.argsort(prob, kind="mergesort")
    s = prob[order]
    ranks_sorted = np.empty(len(s), dtype=float)
    i = 0
    while i < len(s):
        j = i + 1
        while j < len(s) and s[j] == s[i]:
            j += 1
        avg_rank = ((i + 1) + j) / 2.0  # 1-based inclusive average rank
        ranks_sorted[i:j] = avg_rank
        i = j
    ranks = np.empty(len(s), dtype=float)
    ranks[order] = ranks_sorted
    u = float(np.sum(ranks[y == 1]) - n_pos * (n_pos + 1) / 2.0)
    return u / (n_pos * n_neg)


def best_f1_threshold(y: np.ndarray, prob: np.ndarray) -> Tuple[float, float]:
    """Exact validation-set F1 threshold search over observed scores."""
    y = np.asarray(y, int); prob = np.asarray(prob, float)
    if len(y) == 0:
        return 0.5, 0.0
    order = np.argsort(-prob, kind="mergesort")
    ys, ps = y[order], prob[order]
    tp = np.cumsum(ys == 1)
    fp = np.cumsum(ys == 0)
    total_pos = int(np.sum(ys == 1))
    fn = total_pos - tp
    precision = np.divide(tp, tp + fp, out=np.zeros_like(tp, dtype=float), where=(tp + fp) > 0)
    recall = np.divide(tp, tp + fn, out=np.zeros_like(tp, dtype=float), where=(tp + fn) > 0)
    f1 = np.divide(2 * precision * recall, precision + recall,
                   out=np.zeros_like(precision), where=(precision + recall) > 0)
    # Only score positions where the threshold changes.
    ends = np.flatnonzero(np.r_[ps[:-1] != ps[1:], True])
    j = int(ends[np.argmax(f1[ends])])
    return float(np.clip(ps[j], 1e-6, 1.0 - 1e-6)), float(f1[j])


def classification_metrics(y: np.ndarray, prob: np.ndarray, threshold: float) -> Dict[str, float]:
    y = np.asarray(y, int); prob = np.asarray(prob, float)
    pred = (prob >= float(threshold)).astype(int)
    tp, fp, fn, tn = _confusion(y, pred)
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2.0 * prec * rec / (prec + rec) if prec + rec else 0.0
    acc = (tp + tn) / max(1, len(y))
    tpr = rec
    tnr = tn / (tn + fp) if tn + fp else 0.0
    bal = 0.5 * (tpr + tnr)
    denom = math.sqrt(max(0.0, float((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))))
    mcc = ((tp * tn - fp * fn) / denom) if denom > 0 else 0.0
    return {
        "Prec": float(prec), "Recall": float(rec), "F1": float(f1),
        "AUC": float(binary_auc(y, prob)), "MCC": float(mcc),
        "Accuracy": float(acc), "Balanced_Accuracy": float(bal),
    }


def _effort_order(prob: np.ndarray, effort: np.ndarray) -> np.ndarray:
    # Standard JIT effort-aware ranking uses predicted defect density, not raw probability.
    density = np.asarray(prob, float) / np.maximum(np.asarray(effort, float), 1e-12)
    return np.argsort(-density, kind="mergesort")


def _recall_at_effort(y: np.ndarray, effort: np.ndarray, order: np.ndarray, fraction: float) -> float:
    total_defects = float(np.sum(y))
    total_effort = float(np.sum(effort))
    if total_defects <= 0 or total_effort <= 0:
        return 0.0
    ys = y[order]; es = effort[order]
    cum = np.cumsum(es)
    keep = cum <= fraction * total_effort + 1e-12
    return float(np.sum(ys[keep]) / total_defects)


def effort_metrics(y: np.ndarray, prob: np.ndarray, effort: np.ndarray) -> Dict[str, float]:
    y = np.asarray(y, int); prob = np.asarray(prob, float); effort = np.asarray(effort, float)
    effort = np.maximum(effort, 1e-12)
    total_defects = int(np.sum(y)); total_effort = float(np.sum(effort))
    if len(y) == 0 or total_defects == 0 or total_effort <= 0:
        return {"PofB20": 0.0, "R@20%E": 0.0, "E@20%R": 0.0,
                "Popt": 0.0, "Top10": 0.0, "Top5": 0.0, "IFA": float(len(y))}

    pred_order = _effort_order(prob, effort)
    r20 = _recall_at_effort(y, effort, pred_order, 0.20)

    ys = y[pred_order]; es = effort[pred_order]
    target_defects = max(1, int(math.ceil(0.20 * total_defects)))
    bug_cum = np.cumsum(ys)
    hit = np.flatnonzero(bug_cum >= target_defects)
    e20r = float(np.sum(es[:int(hit[0]) + 1]) / total_effort) if len(hit) else 1.0

    # Match the JIT-Fine replication metric: Popt at 10%,20%,...,100% effort.
    optimal_order = np.argsort(-(y / effort), kind="mergesort")
    worst_order = np.argsort(y / effort, kind="mergesort")
    grid = np.arange(0.10, 1.01, 0.10)
    pred_curve = np.asarray([_recall_at_effort(y, effort, pred_order, f) for f in grid])
    opt_curve = np.asarray([_recall_at_effort(y, effort, optimal_order, f) for f in grid])
    worst_curve = np.asarray([_recall_at_effort(y, effort, worst_order, f) for f in grid])
    # NumPy >=2.0 removed np.trapz in favor of np.trapezoid; getattr's default
    # argument is evaluated eagerly, so `getattr(np, "trapezoid", np.trapz)` would
    # itself raise AttributeError on those versions before getattr ever runs. This
    # form only touches whichever name actually exists.
    trap = np.trapezoid if hasattr(np, "trapezoid") else np.trapz
    a_pred = float(trap(pred_curve, grid)); a_opt = float(trap(opt_curve, grid)); a_worst = float(trap(worst_curve, grid))
    denom = a_opt - a_worst
    popt = 1.0 - ((a_opt - a_pred) / denom) if abs(denom) > 1e-12 else 0.0

    top5 = float(np.sum(ys[:5]) / total_defects)
    top10 = float(np.sum(ys[:10]) / total_defects)
    first = np.flatnonzero(ys == 1)
    ifa = float(first[0]) if len(first) else float(len(ys))
    return {
        "PofB20": float(r20), "R@20%E": float(r20), "E@20%R": float(e20r),
        "Popt": float(popt), "Top10": top10, "Top5": top5, "IFA": ifa,
    }


def all_metrics(y: np.ndarray, prob: np.ndarray, effort: np.ndarray, threshold: float) -> Dict[str, float]:
    out = classification_metrics(y, prob, threshold)
    out.update(effort_metrics(y, prob, effort))
    return out


def eval_loader(model: Any, loader: Any, device: str) -> Tuple[np.ndarray, np.ndarray, np.ndarray, List[str]]:
    torch, _, F, _, _ = _torch_modules()
    model.eval()
    ys, ps, es, shas = [], [], [], []
    with torch.no_grad():
        for b in loader:
            for k in ("syntax", "g_inv", "g_type", "g_rel", "g_sign", "g_mask", "process", "semantic", "label", "effort"):
                b[k] = b[k].to(device, non_blocking=True)
            logits = model(b)
            prob = F.softmax(logits, dim=1)[:, 1]
            ys.extend(b["label"].cpu().numpy().tolist())
            ps.extend(prob.cpu().numpy().tolist())
            es.extend(b["effort"].cpu().numpy().tolist())
            shas.extend(b["sha"])
    return np.asarray(ys, int), np.asarray(ps, float), np.asarray(es, float), shas



def _graph_invariants(gt: GraphChangeToken) -> np.ndarray:
    """Rotation-invariant summaries of independently-trained per-graph Node2Vec vectors."""
    zu = np.asarray(gt.z_u, dtype=np.float32)
    zv = np.asarray(gt.z_v, dtype=np.float32)
    nu = float(np.linalg.norm(zu)); nv = float(np.linalg.norm(zv))
    if nu > 1e-12 and nv > 1e-12:
        cos = float(np.dot(zu, zv) / (nu * nv))
    else:
        cos = 0.0
    dist = float(np.linalg.norm(zu - zv))
    return np.asarray([nu, nv, cos, dist, abs(nu - nv), float(nu > 0 and nv > 0)], dtype=np.float32)


def _engineered_three_view_features(items: Sequence[CommitFeatures], n_hash: int = 4096):
    """Sparse, fixed-length representation using all four views (syntax, graph,
    process, pretrained semantic).

    Syntax and graph categorical change patterns are feature-hashed; process metrics and
    graph invariant statistics are appended as numeric features. This complements rather
    than replaces the neural FUSE-JIT representation. The raw semantic embedding is
    returned separately (not concatenated here) because it needs a train-only PCA
    reduction before a tree model can use it usefully -- see _build_three_view_booster_predictions.
    """
    from sklearn.feature_extraction import FeatureHasher
    from scipy import sparse

    docs: List[List[str]] = []
    numeric: List[np.ndarray] = []
    semantic: List[np.ndarray] = []
    for cf in items:
        toks: List[str] = []
        syn_count = 0; graph_count = 0
        graph_inv: List[np.ndarray] = []
        add_n = del_n = add_e = del_e = 0
        rel_counts = collections.Counter()
        for ff in cf.files:
            st = list(ff.syntax_tokens)
            syn_count += len(st)
            # Unigrams + local bigrams retain change identity and short-range order.
            for t in st:
                toks.append("S:" + str(t))
            for a, b in zip(st[:-1], st[1:]):
                toks.append("SB:" + str(a) + "|" + str(b))
            for gt in ff.graph_tokens:
                graph_count += 1
                sign = "A" if int(gt.sign) > 0 else "D"
                toks.append(f"G:{gt.kind}:{sign}:{gt.relation}:{gt.node_type}")
                toks.append(f"GR:{sign}:{gt.relation}")
                toks.append(f"GT:{sign}:{gt.node_type}")
                rel_counts[(sign, str(gt.relation))] += 1
                if gt.kind == "node":
                    if gt.sign > 0: add_n += 1
                    else: del_n += 1
                else:
                    if gt.sign > 0: add_e += 1
                    else: del_e += 1
                graph_inv.append(_graph_invariants(gt))
        docs.append(toks)
        proc = _process_pretransform(cf.process).astype(np.float32)
        if graph_inv:
            gi = np.stack(graph_inv, axis=0)
            gi_stats = np.concatenate([gi.mean(0), gi.std(0), gi.max(0), gi.min(0)]).astype(np.float32)
        else:
            gi_stats = np.zeros(24, dtype=np.float32)
        counts = np.asarray([
            len(cf.files), syn_count, graph_count, add_n, del_n, add_e, del_e,
            rel_counts.get(("A", "cf"), 0), rel_counts.get(("D", "cf"), 0),
            rel_counts.get(("A", "df"), 0), rel_counts.get(("D", "df"), 0),
            rel_counts.get(("A", "call"), 0), rel_counts.get(("D", "call"), 0),
        ], dtype=np.float32)
        counts = np.log1p(np.maximum(counts, 0.0))
        numeric.append(np.concatenate([proc, counts, gi_stats]).astype(np.float32))
        semantic.append(_semantic_vector(cf, SEMANTIC_DIM))

    hasher = FeatureHasher(n_features=int(n_hash), input_type="string", alternate_sign=False)
    Xh = hasher.transform(docs).astype(np.float32).tocsr()
    if Xh.nnz:
        Xh.data = np.log1p(Xh.data)
    Xn = np.stack(numeric, axis=0).astype(np.float32)
    Xs = np.stack(semantic, axis=0).astype(np.float32)
    return Xh, Xn, Xs


def _fit_numeric_scaler(X_train: np.ndarray):
    mean = X_train.mean(0).astype(np.float32)
    std = X_train.std(0).astype(np.float32)
    std[std < 1e-6] = 1.0
    return mean, std


def _apply_numeric_scaler(X: np.ndarray, stats):
    mean, std = stats
    return np.clip((X - mean) / std, -6.0, 6.0).astype(np.float32)


def _build_three_view_booster_predictions(train_cf, val_cf, test_cf, output_dir: Path,
                                           device: str, n_hash: int = 4096):
    """Train/tune XGBoost on TRAIN/VAL only and persist VAL/TEST probabilities.

    Test labels are never used to select booster hyperparameters, blend weights, or threshold.
    """
    cache_npz = output_dir / "three_view_booster_predictions.npz"
    cache_json = output_dir / "three_view_booster_config.json"
    val_shas = np.asarray([x.sha for x in val_cf], dtype=str)
    test_shas = np.asarray([x.sha for x in test_cf], dtype=str)
    # Schema fingerprint over everything that changes what these cached arrays MEAN
    # (not just which commits they cover): a process/semantic schema upgrade with the
    # same val/test sha sets must not silently reuse booster predictions trained on
    # the old, narrower feature set.
    schema_fp = _fingerprint({
        "process_features": list(PROCESS_FEATURES), "semantic_dim": int(SEMANTIC_DIM),
        "n_hash": int(n_hash), "booster_configs": list(globals().get("BOOSTER_CONFIGS", [])),
    })
    if cache_npz.exists():
        try:
            z = np.load(cache_npz, allow_pickle=False)
            cached_schema_fp = str(z["schema_fp"][0]) if "schema_fp" in z else None
            if (cached_schema_fp == schema_fp and np.array_equal(z["val_shas"], val_shas)
                    and np.array_equal(z["test_shas"], test_shas)):
                print("[booster] restored persistent validation/test predictions")
                return z["val_prob"].astype(float), z["test_prob"].astype(float)
            if cached_schema_fp != schema_fp:
                print("[booster] cached predictions predate the current feature schema; retraining.")
        except Exception as ex:
            print(f"[booster] cached predictions ignored: {ex}")

    print("[booster] building complementary process+syntax+graph+semantic features ...")
    Xh_tr, Xn_tr, Xs_tr = _engineered_three_view_features(train_cf, n_hash)
    Xh_va, Xn_va, Xs_va = _engineered_three_view_features(val_cf, n_hash)
    Xh_te, Xn_te, Xs_te = _engineered_three_view_features(test_cf, n_hash)

    # A raw pretrained embedding's individual dimensions are not independently
    # axis-aligned-split-friendly the way engineered features are, so a tree model
    # gets much more out of a train-only PCA reduction than out of the 768 raw
    # dimensions directly (which would also badly outnumber the ~40 engineered
    # numeric features above, relative to ~22K training rows).
    n_pca = int(min(32, Xs_tr.shape[0] - 1, Xs_tr.shape[1]))
    if n_pca >= 2 and np.any(Xs_tr != 0):
        from sklearn.decomposition import PCA
        pca = PCA(n_components=n_pca, random_state=42)
        Ps_tr = pca.fit_transform(Xs_tr).astype(np.float32)
        Ps_va = pca.transform(Xs_va).astype(np.float32)
        Ps_te = pca.transform(Xs_te).astype(np.float32)
    else:
        Ps_tr = np.zeros((Xs_tr.shape[0], 0), dtype=np.float32)
        Ps_va = np.zeros((Xs_va.shape[0], 0), dtype=np.float32)
        Ps_te = np.zeros((Xs_te.shape[0], 0), dtype=np.float32)
    Xn_tr = np.concatenate([Xn_tr, Ps_tr], axis=1)
    Xn_va = np.concatenate([Xn_va, Ps_va], axis=1)
    Xn_te = np.concatenate([Xn_te, Ps_te], axis=1)

    scaler = _fit_numeric_scaler(Xn_tr)
    Xn_tr = _apply_numeric_scaler(Xn_tr, scaler)
    Xn_va = _apply_numeric_scaler(Xn_va, scaler)
    Xn_te = _apply_numeric_scaler(Xn_te, scaler)

    from scipy import sparse
    Xtr = sparse.hstack([Xh_tr, sparse.csr_matrix(Xn_tr)], format="csr")
    Xva = sparse.hstack([Xh_va, sparse.csr_matrix(Xn_va)], format="csr")
    Xte = sparse.hstack([Xh_te, sparse.csr_matrix(Xn_te)], format="csr")
    ytr = np.asarray([x.label for x in train_cf], dtype=int)
    yva = np.asarray([x.label for x in val_cf], dtype=int)
    npos = max(1, int(ytr.sum())); nneg = max(1, int(len(ytr) - npos))
    ratio = nneg / npos

    try:
        from xgboost import XGBClassifier
    except Exception:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xgboost"], check=True)
        from xgboost import XGBClassifier

    best = None
    best_prob = None
    xgb_device = "cuda" if str(device).startswith("cuda") else "cpu"
    configs = list(globals().get("BOOSTER_CONFIGS", []))
    if not configs:
        configs = [{"max_depth": 5, "min_child_weight": 5, "learning_rate": 0.03,
                    "subsample": .9, "colsample_bytree": .9, "scale_mult": .75}]
    for ci, c in enumerate(configs, 1):
        kwargs = dict(
            n_estimators=1500, objective="binary:logistic", eval_metric="aucpr", early_stopping_rounds=60,
            tree_method="hist", device=xgb_device, random_state=42,
            reg_lambda=2.0, reg_alpha=0.05, max_depth=int(c["max_depth"]),
            min_child_weight=float(c["min_child_weight"]), learning_rate=float(c["learning_rate"]),
            subsample=float(c["subsample"]), colsample_bytree=float(c["colsample_bytree"]),
            scale_pos_weight=float(ratio * float(c["scale_mult"])),
            max_bin=256,
        )
        model = XGBClassifier(**kwargs)
        model.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
        pva = model.predict_proba(Xva)[:, 1]
        thr, f1 = best_f1_threshold(yva, pva)
        auc = binary_auc(yva, pva)
        print(f"[booster] config {ci}/{len(configs)} val_F1={f1:.4f} val_AUC={auc:.4f} thr={thr:.4f}")
        key = (f1, auc)
        if best is None or key > best[0]:
            best = (key, ci, c, model, thr)
            best_prob = pva
    _, best_i, best_c, best_model, booster_thr = best
    pte = best_model.predict_proba(Xte)[:, 1]
    tmp = cache_npz.with_suffix('.tmp.npz')
    np.savez_compressed(tmp, val_shas=val_shas, test_shas=test_shas, schema_fp=np.asarray([schema_fp]),
                        val_prob=np.asarray(best_prob, np.float32), test_prob=np.asarray(pte, np.float32))
    os.replace(tmp, cache_npz)
    _atomic_json_dump({"selected_config_index": int(best_i), "selected_config": best_c,
                       "validation_threshold": float(booster_thr), "hash_features": int(n_hash)}, cache_json)
    print(f"[booster] selected config {best_i}; predictions persisted")
    return np.asarray(best_prob, float), np.asarray(pte, float)


def _select_validation_blend(yv: np.ndarray, neural_prob: np.ndarray, booster_prob: np.ndarray,
                             grid_points: int = 41):
    """Jointly select blend weight and threshold on validation F1 only."""
    best = None
    for alpha in np.linspace(0.0, 1.0, int(grid_points)):
        p = alpha * neural_prob + (1.0 - alpha) * booster_prob
        thr, f1 = best_f1_threshold(yv, p)
        auc = binary_auc(yv, p)
        # Prefer simpler neural-heavy solution on exact ties.
        key = (f1, auc, alpha)
        if best is None or key > best[0]:
            best = (key, float(alpha), float(thr), float(f1), float(auc))
    return best[1:]


def train_one_run(project: str, train_cf: Sequence[CommitFeatures], val_cf: Sequence[CommitFeatures],
                  test_cf: Sequence[CommitFeatures], cfg: FeatureConfig, run_idx: int,
                  epochs: int, batch_size: int, lr: float, device: str,
                  weight_decay: float = 1e-4, patience: int = 15,
                  grad_clip: float = 1.0,
                  booster_val_prob: Optional[np.ndarray] = None,
                  booster_test_prob: Optional[np.ndarray] = None,
                  blend_grid_points: int = 41,
                  contrastive_weight: float = 0.15,
                  contrastive_temperature: float = 0.1) -> Dict[str, Any]:
    torch, nn, _, _, DataLoader = _torch_modules()
    seed = 42 + run_idx
    set_global_seed(seed)

    # Stronger previous neural path retained: natural batches + cost-sensitive CE.
    w2v = train_syntax_word2vec(train_cf, cfg, seed)
    vocab = GraphVocab(train_cf)
    process_stats = fit_process_scaler(train_cf)
    train_items = prepare_numpy_items(train_cf, w2v, vocab, cfg, process_stats)
    val_items = prepare_numpy_items(val_cf, w2v, vocab, cfg, process_stats)
    test_items = prepare_numpy_items(test_cf, w2v, vocab, cfg, process_stats)

    pin = str(device).startswith("cuda")
    train_loader = DataLoader(NumpyDataset(train_items), batch_size=batch_size, shuffle=True,
                              num_workers=0, pin_memory=pin, collate_fn=collate_batch)
    val_loader = DataLoader(NumpyDataset(val_items), batch_size=batch_size, shuffle=False,
                            num_workers=0, pin_memory=pin, collate_fn=collate_batch)
    test_loader = DataLoader(NumpyDataset(test_items), batch_size=batch_size, shuffle=False,
                             num_workers=0, pin_memory=pin, collate_fn=collate_batch)

    model = build_model(vocab, cfg).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    n_pos = max(1, int(sum(cf.label for cf in train_cf)))
    n_neg = max(1, len(train_cf) - n_pos)
    class_weight = torch.tensor([1.0, n_neg / n_pos], dtype=torch.float32, device=device)
    criterion = nn.CrossEntropyLoss(weight=class_weight)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=5, min_lr=1e-5)

    best_state = None
    best_val = -1.0
    best_epoch = 0
    stale = 0
    t0 = time.time()
    for epoch in range(1, epochs + 1):
        model.train()
        for b in train_loader:
            for k in ("syntax", "g_inv", "g_type", "g_rel", "g_sign", "g_mask", "process", "semantic", "label"):
                b[k] = b[k].to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            logits, fm = model(b, return_embedding=True)
            loss = criterion(logits, b["label"])
            if contrastive_weight > 0:
                # Auxiliary supervised-contrastive term on the fused embedding: sharpens
                # class separation on top of the classification loss (see
                # supervised_contrastive_loss docstring for the rationale).
                loss = loss + contrastive_weight * supervised_contrastive_loss(
                    torch, fm, b["label"], temperature=contrastive_temperature)
            loss.backward()
            if grad_clip and grad_clip > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            opt.step()

        yv, pv, _, _ = eval_loader(model, val_loader, device)
        # Checkpointing/scheduling target a 50/50 blend of F1 and AUC -- the two
        # headline metrics -- rather than F1 alone, since a checkpoint chosen purely
        # for F1 can leave ranking quality (AUC) on the table.
        _, fv = best_f1_threshold(yv, pv)
        auc_v = binary_auc(yv, pv)
        score_v = 0.5 * fv + 0.5 * auc_v
        scheduler.step(score_v)
        if score_v > best_val + 1e-6:
            best_val = score_v
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            stale = 0
        else:
            stale += 1
        if epoch >= 10 and stale >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    yv, pv_neural, ev, val_shas = eval_loader(model, val_loader, device)
    yt, pt_neural, et, shas = eval_loader(model, test_loader, device)
    _, neural_val_f1_final = best_f1_threshold(yv, pv_neural)
    neural_val_auc_final = binary_auc(yv, pv_neural)

    alpha = 1.0
    if booster_val_prob is not None and booster_test_prob is not None:
        if len(booster_val_prob) != len(yv) or len(booster_test_prob) != len(yt):
            raise RuntimeError("Booster probability lengths do not match validation/test partitions.")
        alpha, best_thr, blend_val_f1, blend_val_auc = _select_validation_blend(
            yv, pv_neural, np.asarray(booster_val_prob, float), blend_grid_points)
        pt = alpha * pt_neural + (1.0 - alpha) * np.asarray(booster_test_prob, float)
        pv_blend = alpha * pv_neural + (1.0 - alpha) * np.asarray(booster_val_prob, float)
    else:
        best_thr, blend_val_f1 = best_f1_threshold(yv, pv_neural)
        blend_val_auc = binary_auc(yv, pv_neural)
        pt = pt_neural
        pv_blend = pv_neural

    pred = (pt >= best_thr).astype(int)
    metrics = all_metrics(yt, pt, et, best_thr)
    project_by_sha = {cf.sha: cf.project for cf in test_cf}
    val_project_by_sha = {cf.sha: cf.project for cf in val_cf}

    per_project_metrics: List[Dict[str, Any]] = []
    projects_arr = np.asarray([project_by_sha.get(s, "") for s in shas], dtype=object)
    for p in sorted(set(projects_arr.tolist())):
        m = projects_arr == p
        if np.any(m):
            pm = all_metrics(yt[m], pt[m], et[m], best_thr)
            per_project_metrics.append({"project": p, "run": run_idx + 1, "n_test": int(np.sum(m)), **pm})

    result: Dict[str, Any] = {
        "project": project, "run": run_idx + 1, "seed": seed,
        "n_train": len(train_cf), "n_val": len(val_cf), "n_test": len(test_cf),
        "train_defective": int(sum(x.label for x in train_cf)),
        "val_defective": int(sum(x.label for x in val_cf)),
        "test_defective": int(sum(x.label for x in test_cf)),
        "best_epoch": best_epoch, "threshold": best_thr,
        "class_weight_pos": float(n_neg / n_pos),
        "neural_val_f1": float(neural_val_f1_final), "neural_val_auc": float(neural_val_auc_final),
        "neural_val_checkpoint_score": float(best_val),
        "blend_val_f1": float(blend_val_f1),
        "blend_val_auc": float(blend_val_auc), "neural_blend_weight": float(alpha),
        "runtime_sec": time.time() - t0,
        **metrics,
        "test_predictions": [
            {"sha": s, "project": project_by_sha.get(s, ""), "label": int(y),
             "prob": float(p), "neural_prob": float(pn),
             "booster_prob": float(pb) if booster_test_prob is not None else None,
             "pred": int(pr), "effort": float(e), "defect_density": float(p / max(e, 1e-12))}
            for s, y, p, pn, pb, pr, e in zip(
                shas, yt, pt, pt_neural,
                np.asarray(booster_test_prob, float) if booster_test_prob is not None else np.full(len(pt), np.nan),
                pred, et)
        ],
        # Validation-set probabilities are persisted alongside the test predictions so
        # a post-hoc cross-run ensemble (bagging over independently seeded runs on the
        # same split) can pick its weights/threshold from validation only, never test.
        "val_predictions": [
            {"sha": s, "project": val_project_by_sha.get(s, ""), "label": int(y),
             "prob": float(p), "neural_prob": float(pn),
             "booster_prob": float(pb) if booster_val_prob is not None else None}
            for s, y, p, pn, pb in zip(
                val_shas, yv, pv_blend, pv_neural,
                np.asarray(booster_val_prob, float) if booster_val_prob is not None else np.full(len(pv_blend), np.nan))
        ],
        "per_project_metrics": per_project_metrics,
    }
    return result


# ------------------------------- commands -----------------------------------

# --------------------- multiprocessing + persistent checkpoints ------------------

_PREPARE_WORKER_REPOS: Dict[str, GitRepo] = {}
_PREPARE_WORKER_MAX_FILES = 100
_PREPARE_WORKER_MAX_CHANGED_LOC = 10_000


def _atomic_pickle_dump(obj: Any, path: Path) -> None:
    """Write a pickle atomically so a runtime loss cannot corrupt a checkpoint."""
    ensure_dir(path.parent)
    tmp = path.with_name(path.name + f".tmp.{os.getpid()}")
    with tmp.open("wb") as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)


def _atomic_json_dump(obj: Any, path: Path) -> None:
    ensure_dir(path.parent)
    tmp = path.with_name(path.name + f".tmp.{os.getpid()}")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)


def _atomic_csv_write(rows: Sequence[Mapping[str, Any]], path: Path, fieldnames: Sequence[str]) -> None:
    ensure_dir(path.parent)
    tmp = path.with_name(path.name + f".tmp.{os.getpid()}")
    with tmp.open("w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(fieldnames))
        w.writeheader()
        w.writerows(rows)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)


def _atomic_copy_file(src: Path, dst: Path) -> None:
    ensure_dir(dst.parent)
    tmp = dst.with_name(dst.name + f".tmp.{os.getpid()}")
    shutil.copy2(src, tmp)
    os.replace(tmp, dst)


def _fingerprint(obj: Any) -> str:
    raw = json.dumps(obj, sort_keys=True, separators=(",", ":"), ensure_ascii=True).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()


def _load_or_validate_manifest(stage_dir: Path, manifest: Mapping[str, Any]) -> None:
    """Never mix checkpoints created from a different dataset/configuration.

    Earlier behavior raised and asked the user to manually rename/delete the stage
    checkpoint directory on Google Drive. This is automatic and non-destructive
    instead: on a fingerprint mismatch (e.g. a process-metric schema upgrade), the
    old batch shards and manifest are moved into a timestamped `_stale_*` archive
    subdirectory -- never deleted -- and a fresh manifest is written so the stage
    naturally reprocesses everything under the new fingerprint on this same run.
    """
    ensure_dir(stage_dir)
    path = stage_dir / "manifest.json"
    if not path.exists():
        _atomic_json_dump(dict(manifest), path)
        return
    old = json.loads(path.read_text(encoding="utf-8"))
    if old.get("fingerprint") == manifest.get("fingerprint"):
        return
    archive = stage_dir / f"_stale_{str(old.get('fingerprint', 'unknown'))[:12]}_{int(time.time())}"
    ensure_dir(archive)
    for p in list(stage_dir.glob("batch_*.pkl")) + [path]:
        if p.exists():
            shutil.move(str(p), str(archive / p.name))
    print(f"[checkpoint] {stage_dir}: configuration/schema changed vs. the previous checkpointed "
          f"run; archived old shards to {archive} (nothing deleted) and starting a fresh "
          "checkpoint generation for this stage.")
    _atomic_json_dump(dict(manifest), path)


def _next_batch_number(stage_dir: Path) -> int:
    nums: List[int] = []
    for p in stage_dir.glob("batch_*.pkl"):
        m = re.match(r"batch_(\d+)\.pkl$", p.name)
        if m:
            nums.append(int(m.group(1)))
    return (max(nums) + 1) if nums else 1


def _deadline_reached(deadline: Optional[float]) -> bool:
    # `deadline and ...` would short-circuit on a falsy 0.0 and wrongly report "no
    # deadline" for that value; deadlines are always time.time()+hours in practice
    # (never exactly 0.0), but `is not None` is the correct check regardless.
    return deadline is not None and time.time() >= float(deadline)


def _cleanup_checkpoint_batches(stage_dir: Path) -> None:
    """KEEP all completed persistent checkpoint shards for maximum recoverability.

    The earlier version deleted batch_*.pkl after a final cache was written.
    This safer version intentionally retains them so Stage 1/2 can still be
    reconstructed from checkpoints if the final cache is ever missing/corrupt.
    Only abandoned temporary files are removed.
    """
    removed_tmp = 0
    for p in stage_dir.glob("*.tmp.*"):
        try:
            p.unlink()
            removed_tmp += 1
        except OSError:
            pass
    print(f"[checkpoint] keeping persistent batch shards in {stage_dir}")
    if removed_tmp:
        print(f"[checkpoint] removed {removed_tmp} abandoned temporary files only")


def _prepare_worker_init(repos_root: str, max_files: int, max_changed_loc: int) -> None:
    """Initialize read-only Git repository handles once per worker process."""
    global _PREPARE_WORKER_REPOS, _PREPARE_WORKER_MAX_FILES, _PREPARE_WORKER_MAX_CHANGED_LOC
    _PREPARE_WORKER_REPOS = load_repos(Path(repos_root))
    _PREPARE_WORKER_MAX_FILES = int(max_files)
    _PREPARE_WORKER_MAX_CHANGED_LOC = int(max_changed_loc)


def _prepare_worker(task: Tuple[str, str, str, int]) -> Tuple[str, Optional[CommitRecord], Optional[Dict[str, Any]]]:
    """Reconstruct one already-resolved commit using the unchanged core routine."""
    key, project, full_sha, label = task
    rec = build_commit_record(
        _PREPARE_WORKER_REPOS[project], full_sha, label,
        _PREPARE_WORKER_MAX_FILES, _PREPARE_WORKER_MAX_CHANGED_LOC,
    )
    if rec is None:
        skipped = {
            "sha": full_sha, "project": project, "label": label,
            "reason": "no usable Java pair/root/extreme",
        }
        return key, None, skipped
    return key, rec, None


def _fork_context() -> Any:
    """Colab is Linux; explicit fork preserves deterministic process state."""
    try:
        return mp.get_context("fork")
    except ValueError:
        return mp.get_context()


def _load_prepare_checkpoints(stage_dir: Path, fingerprint: str) -> Dict[str, Tuple[Optional[CommitRecord], Optional[Dict[str, Any]]]]:
    done: Dict[str, Tuple[Optional[CommitRecord], Optional[Dict[str, Any]]]] = {}
    for p in sorted(stage_dir.glob("batch_*.pkl")):
        try:
            with p.open("rb") as f:
                obj = pickle.load(f)
            if obj.get("fingerprint") != fingerprint:
                raise RuntimeError(f"Checkpoint fingerprint mismatch: {p}")
            for key, rec, skip in obj.get("items", []):
                done[str(key)] = (rec, skip)
        except Exception as ex:
            raise RuntimeError(f"Could not read persistent prepare checkpoint {p}: {ex}") from ex
    return done


def cmd_prepare(args: argparse.Namespace) -> bool:
    dataset_root = Path(args.dataset_root).resolve()
    repos_root = Path(args.repos_root).resolve()
    cache_dir = ensure_dir(Path(args.cache_dir).resolve())
    persistent_cache_dir = ensure_dir(Path(getattr(args, "persistent_cache_dir", args.cache_dir)).resolve())
    stage_dir = ensure_dir(persistent_cache_dir / "checkpoints" / "prepare")
    checkpoint_every = max(1, int(getattr(args, "checkpoint_every", 200)))
    deadline = getattr(args, "session_deadline", None)

    if args.clone_repos:
        clone_repositories(repos_root)
    repos = load_repos(repos_root)
    labels = read_labels_csv(Path(args.labels_csv)) if args.labels_csv else discover_labels(dataset_root)
    if args.labels_csv:
        save_json([], cache_dir / "label_conflicts.json")
    else:
        save_json(LAST_LABEL_CONFLICTS, cache_dir / "label_conflicts.json")
    print(f"[labels] discovered {len(labels):,} unique commit identifiers")
    if LAST_LABEL_CONFLICTS and not args.labels_csv:
        print(f"[labels] conflict audit: {cache_dir / 'label_conflicts.json'}")

    resolver = ShaResolver(repos)
    unresolved: List[Dict[str, Any]] = []
    tasks: List[Tuple[str, str, str, int]] = []
    for e in labels.values():
        project, full = map_sha_to_repo(e, repos, resolver)
        if not project or not full:
            unresolved.append({"sha": e.sha, "label": e.label, "hint": e.project_hint, "sources": e.source_paths[:3]})
            continue
        key = f"{project}|{full}|{int(e.label)}"
        tasks.append((key, project, full, int(e.label)))

    # Fingerprint is order-independent so filesystem traversal order cannot invalidate a resume.
    # "process_features" is included so that adding/renaming a process metric (e.g. the
    # AGE/EXP/REXP/SEXP experience metrics) automatically invalidates stale checkpoint
    # shards instead of silently mixing old- and new-schema CommitRecords.
    fp_payload = {
        "stage": "prepare-v3-experience-metrics",
        "tasks": sorted((p, s, int(y)) for _, p, s, y in tasks),
        "max_files": int(args.max_files),
        "max_changed_loc": int(args.max_changed_loc),
        "process_features": list(PROCESS_FEATURES),
    }
    fp = _fingerprint(fp_payload)
    _load_or_validate_manifest(stage_dir, {
        "fingerprint": fp,
        "stage": "prepare",
        "total_resolved": len(tasks),
        "max_files": int(args.max_files),
        "max_changed_loc": int(args.max_changed_loc),
    })

    done = _load_prepare_checkpoints(stage_dir, fp)
    remaining = [t for t in tasks if t[0] not in done]
    workers = max(1, int(getattr(args, "prepare_workers", 1)))
    print(f"[prepare] CPU workers={workers}; resolved={len(tasks):,}; unresolved={len(unresolved):,}")
    if done:
        usable_done = sum(1 for rec, _ in done.values() if rec is not None)
        print(f"[prepare] RESUME: recovered {len(done):,}/{len(tasks):,} completed commits "
              f"({usable_done:,} usable) from Google Drive; remaining={len(remaining):,}")

    buffer: List[Tuple[str, Optional[CommitRecord], Optional[Dict[str, Any]]]] = []
    batch_no = _next_batch_number(stage_dir)
    completed = len(done)

    def flush_buffer() -> None:
        nonlocal buffer, batch_no
        if not buffer:
            return
        path = stage_dir / f"batch_{batch_no:06d}.pkl"
        _atomic_pickle_dump({"fingerprint": fp, "items": buffer}, path)
        for key, rec, skip in buffer:
            done[key] = (rec, skip)
        print(f"[prepare] persistent checkpoint -> {path.name} ({len(buffer)} commits)")
        batch_no += 1
        buffer = []

    stopped_for_deadline = False
    if workers == 1:
        for key, project, full, label in remaining:
            if _deadline_reached(deadline):
                stopped_for_deadline = True
                break
            rec = build_commit_record(repos[project], full, label, args.max_files, args.max_changed_loc)
            skip = None if rec is not None else {
                "sha": full, "project": project, "label": label,
                "reason": "no usable Java pair/root/extreme",
            }
            buffer.append((key, rec, skip))
            completed += 1
            if len(buffer) >= checkpoint_every:
                flush_buffer()
            if completed % 100 == 0 or completed == len(tasks):
                usable = sum(1 for rec0, _ in done.values() if rec0 is not None) + sum(1 for _, rec0, _ in buffer if rec0 is not None)
                print(f"[prepare] {completed:,}/{len(tasks):,} resolved commits; usable={usable:,}")
    else:
        ctx = _fork_context()
        pool = ctx.Pool(
            processes=workers,
            initializer=_prepare_worker_init,
            initargs=(str(repos_root), args.max_files, args.max_changed_loc),
        )
        try:
            for key, rec, skip in pool.imap_unordered(_prepare_worker, remaining, chunksize=1):
                buffer.append((key, rec, skip))
                completed += 1
                if len(buffer) >= checkpoint_every:
                    flush_buffer()
                if completed % 100 == 0 or completed == len(tasks):
                    usable = sum(1 for rec0, _ in done.values() if rec0 is not None) + sum(1 for _, rec0, _ in buffer if rec0 is not None)
                    print(f"[prepare] {completed:,}/{len(tasks):,} resolved commits; usable={usable:,}")
                if _deadline_reached(deadline):
                    stopped_for_deadline = True
                    break
            if stopped_for_deadline:
                pool.terminate()
            else:
                pool.close()
        finally:
            pool.join()

    flush_buffer()

    if stopped_for_deadline or len(done) < len(tasks):
        print(f"[prepare] SAFE STOP: {len(done):,}/{len(tasks):,} commits are persistently checkpointed.")
        print("[prepare] Start a new Colab runtime and run the SAME script; it will continue the remaining commits.")
        return False

    records = [rec for rec, _ in done.values() if rec is not None]
    skipped = [skip for _, skip in done.values() if skip is not None]
    records.sort(key=lambda r: (r.project, r.author_date, r.sha))
    unresolved.sort(key=lambda x: (str(x.get("sha", "")), int(x.get("label", 0))))
    skipped.sort(key=lambda x: (str(x.get("project", "")), str(x.get("sha", "")), int(x.get("label", 0))))

    local_records = cache_dir / "commit_records.pkl"
    _atomic_pickle_dump(records, local_records)
    # Persist the completed Stage-1 cache. Checkpoint shards remain as a recovery fallback.
    _atomic_pickle_dump(records, persistent_cache_dir / "commit_records.pkl")
    save_json(unresolved, cache_dir / "unresolved_labels.json")
    save_json(skipped, cache_dir / "skipped_commits.json")
    stats = []
    for p in PROJECT_REPOS:
        rs = [r for r in records if r.project == p]
        if not rs:
            continue
        stats.append({
            "project": p, "commits_after_reconstruction": len(rs),
            "defective": sum(r.label for r in rs), "clean": len(rs) - sum(r.label for r in rs),
            "defect_rate": sum(r.label for r in rs) / len(rs),
            "start": min(r.author_date for r in rs), "end": max(r.author_date for r in rs),
        })
    if stats:
        _atomic_csv_write(stats, cache_dir / "dataset_stats.csv", list(stats[0].keys()))
    for name in ("label_conflicts.json", "unresolved_labels.json", "skipped_commits.json", "dataset_stats.csv"):
        src = cache_dir / name
        if src.exists():
            _atomic_copy_file(src, persistent_cache_dir / name)
    _cleanup_checkpoint_batches(stage_dir)
    print(f"[prepare] saved {len(records):,} usable commit records locally AND persistently")
    print(f"[prepare] unresolved labels: {len(unresolved):,}; skipped after mapping: {len(skipped):,}")
    return True


def load_commit_records(cache_dir: Path) -> List[CommitRecord]:
    p = cache_dir / "commit_records.pkl"
    if not p.exists():
        raise FileNotFoundError(f"Missing {p}; run the prepare command first.")
    with p.open("rb") as f:
        raw = pickle.load(f)
    if raw and isinstance(raw[0], CommitRecord):
        return raw
    return [CommitRecord.from_dict(x) for x in raw]


_FEAT_WORKER_RECORDS: Optional[Sequence[CommitRecord]] = None
_FEAT_WORKER_PARSER: Optional[JavaParser] = None
_FEAT_WORKER_CFG: Optional[FeatureConfig] = None


def _featurize_worker_init(cfg: FeatureConfig) -> None:
    """Create one Tree-sitter parser per process; records are inherited by fork."""
    global _FEAT_WORKER_PARSER, _FEAT_WORKER_CFG
    _FEAT_WORKER_PARSER = JavaParser()
    _FEAT_WORKER_CFG = cfg


def _featurize_worker(index: int) -> Tuple[int, Optional[CommitFeatures], Optional[str]]:
    records = _FEAT_WORKER_RECORDS
    parser = _FEAT_WORKER_PARSER
    cfg = _FEAT_WORKER_CFG
    if records is None or parser is None or cfg is None:
        raise RuntimeError("Feature worker was not initialized correctly")
    r = records[index]
    try:
        return index, featurize_record(r, parser, cfg), None
    except Exception as ex:
        return index, None, f"[WARN] feature extraction failed for {r.project} {r.sha}: {ex}"


def _load_feature_checkpoints(stage_dir: Path, fingerprint: str) -> Dict[int, Tuple[Optional[CommitFeatures], Optional[str]]]:
    done: Dict[int, Tuple[Optional[CommitFeatures], Optional[str]]] = {}
    for p in sorted(stage_dir.glob("batch_*.pkl")):
        try:
            with p.open("rb") as f:
                obj = pickle.load(f)
            if obj.get("fingerprint") != fingerprint:
                raise RuntimeError(f"Checkpoint fingerprint mismatch: {p}")
            for idx, feature, warning in obj.get("items", []):
                done[int(idx)] = (feature, warning)
        except Exception as ex:
            raise RuntimeError(f"Could not read persistent feature checkpoint {p}: {ex}") from ex
    return done


def cmd_featurize(args: argparse.Namespace) -> bool:
    cache_dir = Path(args.cache_dir).resolve()
    persistent_cache_dir = ensure_dir(Path(getattr(args, "persistent_cache_dir", args.cache_dir)).resolve())
    stage_dir = ensure_dir(persistent_cache_dir / "checkpoints" / "featurize")
    checkpoint_every = max(1, int(getattr(args, "checkpoint_every", 100)))
    deadline = getattr(args, "session_deadline", None)
    records = load_commit_records(cache_dir)
    cfg = FeatureConfig(
        syntax_dim=args.syntax_dim, syntax_window=args.syntax_window, seq_len=args.seq_len,
        graph_dim=args.graph_dim, graph_window=args.graph_window,
        walk_length=args.walk_length, num_walks=args.num_walks,
        node2vec_p=args.node2vec_p, node2vec_q=args.node2vec_q,
    )
    out_path = cache_dir / "commit_features.pkl"
    workers = max(1, int(getattr(args, "featurize_workers", 1)))

    # "process_features" guards against a subtler staleness bug: this fingerprint's
    # record identity tuple (project, sha, label, author_date) does not change when
    # Stage 1 is regenerated with a different process-metric schema, so without this
    # an existing Stage-2 checkpoint shard would otherwise be silently reused with
    # stale (shorter) process vectors baked into its cached CommitFeatures.
    fp_payload = {
        "stage": "featurize-v3-experience-metrics",
        "records": [(r.project, r.sha, int(r.label), r.author_date) for r in records],
        "config": dataclasses.asdict(cfg),
        "process_features": list(PROCESS_FEATURES),
    }
    fp = _fingerprint(fp_payload)
    _load_or_validate_manifest(stage_dir, {
        "fingerprint": fp,
        "stage": "featurize",
        "total_records": len(records),
        "config": dataclasses.asdict(cfg),
    })
    done = _load_feature_checkpoints(stage_dir, fp)
    remaining = [i for i in range(len(records)) if i not in done]
    print(f"[features] CPU workers={workers}; commits={len(records):,}")
    if done:
        successful_done = sum(1 for f, _ in done.values() if f is not None)
        print(f"[features] RESUME: recovered {len(done):,}/{len(records):,} completed commits "
              f"({successful_done:,} successful) from Google Drive; remaining={len(remaining):,}")

    buffer: List[Tuple[int, Optional[CommitFeatures], Optional[str]]] = []
    batch_no = _next_batch_number(stage_dir)
    completed = len(done)

    def flush_buffer() -> None:
        nonlocal buffer, batch_no
        if not buffer:
            return
        path = stage_dir / f"batch_{batch_no:06d}.pkl"
        _atomic_pickle_dump({"fingerprint": fp, "items": buffer}, path)
        for idx, feature, warning in buffer:
            done[int(idx)] = (feature, warning)
        print(f"[features] persistent checkpoint -> {path.name} ({len(buffer)} commits)")
        batch_no += 1
        buffer = []

    stopped_for_deadline = False
    if workers == 1:
        parser = JavaParser()
        for i in remaining:
            if _deadline_reached(deadline):
                stopped_for_deadline = True
                break
            r = records[i]
            try:
                feature = featurize_record(r, parser, cfg)
                warning = None
            except Exception as ex:
                feature = None
                warning = f"[WARN] feature extraction failed for {r.project} {r.sha}: {ex}"
                print(warning, file=sys.stderr)
            buffer.append((i, feature, warning))
            completed += 1
            if len(buffer) >= checkpoint_every:
                flush_buffer()
            if completed % 50 == 0 or completed == len(records):
                successful = sum(1 for f, _ in done.values() if f is not None) + sum(1 for _, f, _ in buffer if f is not None)
                print(f"[features] {completed:,}/{len(records):,}; successful={successful:,}")
    else:
        global _FEAT_WORKER_RECORDS
        _FEAT_WORKER_RECORDS = records
        ctx = _fork_context()
        pool = ctx.Pool(
            processes=workers,
            initializer=_featurize_worker_init,
            initargs=(cfg,),
        )
        try:
            for idx, feature, warning in pool.imap_unordered(_featurize_worker, remaining, chunksize=1):
                if warning:
                    print(warning, file=sys.stderr)
                buffer.append((idx, feature, warning))
                completed += 1
                if len(buffer) >= checkpoint_every:
                    flush_buffer()
                if completed % 50 == 0 or completed == len(records):
                    successful = sum(1 for f, _ in done.values() if f is not None) + sum(1 for _, f, _ in buffer if f is not None)
                    print(f"[features] {completed:,}/{len(records):,}; successful={successful:,}")
                if _deadline_reached(deadline):
                    stopped_for_deadline = True
                    break
            if stopped_for_deadline:
                pool.terminate()
            else:
                pool.close()
        finally:
            pool.join()
            _FEAT_WORKER_RECORDS = None

    flush_buffer()

    if stopped_for_deadline or len(done) < len(records):
        print(f"[features] SAFE STOP: {len(done):,}/{len(records):,} commits are persistently checkpointed.")
        print("[features] Start a new Colab runtime and run the SAME script; Stage 2 will resume only the remainder.")
        return False

    # Indexed checkpoints restore the exact original record order regardless of worker completion order.
    features: List[CommitFeatures] = []
    failures = 0
    for i in range(len(records)):
        feature, warning = done[i]
        if feature is not None:
            features.append(feature)
        else:
            failures += 1
    final_obj = {"config": dataclasses.asdict(cfg), "features": features}
    _atomic_pickle_dump(final_obj, out_path)
    _atomic_pickle_dump(final_obj, persistent_cache_dir / "commit_features.pkl")
    _cleanup_checkpoint_batches(stage_dir)
    print(f"[features] saved {len(features):,} commits locally AND persistently; failed={failures:,}")
    return True


def load_commit_features(cache_dir: Path) -> Tuple[FeatureConfig, List[CommitFeatures]]:
    p = cache_dir / "commit_features.pkl"
    if not p.exists():
        raise FileNotFoundError(f"Missing {p}; run the featurize command first.")
    with p.open("rb") as f:
        obj = pickle.load(f)
    if isinstance(obj, dict) and "features" in obj:
        cfg = FeatureConfig(**obj["config"])
        return cfg, obj["features"]
    return FeatureConfig(), obj


# ------------------- Stage 2.5: pretrained semantic embeddings ---------------
#
# Deliberately decoupled from cmd_featurize: it reads commit_records.pkl directly
# (only needs raw old/new file text, not any tree-sitter/Node2Vec output) and is a
# single-process GPU-batched pass rather than a CPU multiprocessing one, which is
# the right execution model for transformer inference. This also means it never
# needs to repeat the expensive AST/graph extraction when only the semantic model
# or its config changes.

def _semantic_model_bundle(model_name: str, device: str):
    """Load a pretrained code-model tokenizer/encoder once per process."""
    try:
        from transformers import AutoTokenizer, AutoModel
    except Exception as ex:
        raise RuntimeError(
            "Semantic embeddings require the `transformers` package (pip install transformers)."
        ) from ex
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    model.to(device)
    model.eval()
    return tokenizer, model


def _encode_texts_batch(texts: Sequence[str], tokenizer: Any, model: Any, device: str,
                        max_length: int, use_fp16: bool) -> np.ndarray:
    import contextlib
    torch, _, _, _, _ = _torch_modules()
    # An empty diff text (e.g. a whitespace/comment-only change) still needs a valid,
    # non-empty input for the tokenizer; a single pad-like placeholder keeps the batch
    # shape uniform without special-casing empty rows downstream.
    safe_texts = [t if t.strip() else "<EMPTY>" for t in texts]
    enc = tokenizer(safe_texts, padding=True, truncation=True, max_length=max_length,
                    return_tensors="pt")
    enc = {k: v.to(device) for k, v in enc.items()}
    autocast_ctx = (
        torch.autocast(device_type="cuda", dtype=torch.float16)
        if use_fp16 and str(device).startswith("cuda") else contextlib.nullcontext()
    )
    with torch.no_grad(), autocast_ctx:
        out = model(**enc)
        cls = out.last_hidden_state[:, 0, :]  # pretrained [CLS]/<s> pooled representation
    return cls.detach().float().cpu().numpy()


def extract_semantic_embeddings(records: Sequence["CommitRecord"], features: Sequence[CommitFeatures],
                                stage_root: Path, device: str,
                                model_name: str = "microsoft/codebert-base",
                                max_length: int = 256, batch_size: int = 64,
                                max_diff_chars: int = 6000, use_fp16: bool = True,
                                checkpoint_every: int = 2000,
                                session_deadline: Optional[float] = None) -> bool:
    """Populate CommitFeatures.semantic_emb in place via a pretrained code model.

    Runs ONCE for the whole dataset (not once per neural training run) and is
    independently checkpointed/resumable like Stage 1/2, since extraction over the
    full 21-project dataset can still be interrupted by a Colab disconnect. Returns
    True if every commit now has an embedding, False if a safe stop was hit (call
    again, with the SAME cached records/features, to resume).
    """
    stage_dir = ensure_dir(stage_root / "checkpoints" / "semantic")
    fp_payload = {
        "stage": "semantic-v1", "model_name": model_name, "max_length": max_length,
        "max_diff_chars": max_diff_chars,
        "keys": sorted((cf.project, cf.sha) for cf in features),
    }
    fp = _fingerprint(fp_payload)
    _load_or_validate_manifest(stage_dir, {"fingerprint": fp, "stage": "semantic"})

    done: Dict[Tuple[str, str], np.ndarray] = {}
    for p in sorted(stage_dir.glob("batch_*.pkl")):
        with p.open("rb") as f:
            obj = pickle.load(f)
        if obj.get("fingerprint") == fp:
            done.update(obj.get("items", {}))

    remaining_keys = [(cf.project, cf.sha) for cf in features if (cf.project, cf.sha) not in done]
    if done:
        print(f"[semantic] RESUME: recovered {len(done):,}/{len(features):,} embeddings; "
              f"remaining={len(remaining_keys):,}")
    if not remaining_keys:
        for cf in features:
            cf.semantic_emb = done[(cf.project, cf.sha)]
        _cleanup_checkpoint_batches(stage_dir)
        return True

    record_by_key = {(r.project, r.sha): r for r in records}
    print(f"[semantic] extracting {model_name} embeddings for {len(remaining_keys):,} commits "
          f"(device={device}, batch_size={batch_size})")
    tokenizer, model = _semantic_model_bundle(model_name, device)

    buffer: Dict[Tuple[str, str], np.ndarray] = {}
    batch_no = _next_batch_number(stage_dir)
    completed = len(done)
    finished = True
    for i in range(0, len(remaining_keys), batch_size):
        if _deadline_reached(session_deadline):
            finished = False
            break
        chunk_keys = remaining_keys[i:i + batch_size]
        texts = [build_change_diff_text(record_by_key[k], max_diff_chars) for k in chunk_keys]
        embs = _encode_texts_batch(texts, tokenizer, model, device, max_length, use_fp16)
        for key, emb in zip(chunk_keys, embs):
            buffer[key] = emb.astype(np.float32)
        completed += len(chunk_keys)
        if len(buffer) >= checkpoint_every:
            path = stage_dir / f"batch_{batch_no:06d}.pkl"
            _atomic_pickle_dump({"fingerprint": fp, "items": buffer}, path)
            done.update(buffer)
            print(f"[semantic] checkpoint -> {path.name} ({len(buffer)} commits); "
                  f"{completed:,}/{len(features):,} total")
            batch_no += 1
            buffer = {}
        elif completed % 5000 < batch_size:
            print(f"[semantic] {completed:,}/{len(features):,} embedded so far")
    if buffer:
        path = stage_dir / f"batch_{batch_no:06d}.pkl"
        _atomic_pickle_dump({"fingerprint": fp, "items": buffer}, path)
        done.update(buffer)
        print(f"[semantic] checkpoint -> {path.name} ({len(buffer)} commits)")

    for cf in features:
        key = (cf.project, cf.sha)
        if key in done:
            cf.semantic_emb = done[key]

    if finished:
        _cleanup_checkpoint_batches(stage_dir)
        print(f"[semantic] extraction complete for {len(features):,} commits")
    else:
        print(f"[semantic] SAFE STOP: {len(done):,}/{len(features):,} embeddings checkpointed; "
              "rerun with the same cache to resume.")
    return finished


def _load_existing_run_rows(path: Path) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    if not path.exists():
        return rows
    with path.open("r", encoding="utf-8", newline="") as f:
        for row in csv.DictReader(f):
            if not row:
                continue
            out: Dict[str, Any] = dict(row)
            for k in ("run", "seed", "n_train", "n_val", "n_test", "train_defective", "val_defective", "test_defective", "best_epoch"):
                if k in out and str(out[k]).strip() != "":
                    out[k] = int(float(out[k]))
            for k in ("threshold", "class_weight_pos", "runtime_sec", *METRIC_NAMES):
                if k in out and str(out[k]).strip() != "":
                    out[k] = float(out[k])
            rows.append(out)
    return rows


def _load_csv_dicts(path: Path) -> List[Dict[str, Any]]:
    if not path.exists():
        return []
    with path.open("r", encoding="utf-8", newline="") as f:
        return [dict(r) for r in csv.DictReader(f) if r]


def _load_ensemble_matrices(pred_dir: Path, indices: Sequence[int]) -> Optional[Tuple[Any, ...]]:
    """Load and sha-align validation/test prediction matrices across completed runs.

    Every run trains on the SAME fixed 8:1:1 split (only the model's random init
    differs across runs), so averaging independently-seeded runs' final blended
    probabilities is a valid bagging ensemble that needs no extra validation data
    and cannot leak test information: each run's own alpha/threshold were already
    selected on validation only, and this step only ever re-selects a threshold
    (and, for the stacker, weights) on the aligned validation matrix.

    A run index is only included when BOTH its val_run_%02d.json and run_%02d.json
    are present. This deliberately (and silently) excludes runs computed before a
    process/config schema upgrade added the val_run file, so the bagging pool never
    silently mixes predictions from two incompatible feature schemas.
    Returns None if fewer than 2 runs qualify.
    """
    usable = [i for i in indices
              if (pred_dir / f"val_run_{i:02d}.json").exists() and (pred_dir / f"run_{i:02d}.json").exists()]
    if len(usable) < 2:
        return None

    def _matrix(lists: Sequence[List[Dict[str, Any]]], shas: Sequence[str]) -> np.ndarray:
        mat = np.zeros((len(lists), len(shas)), dtype=float)
        for i, plist in enumerate(lists):
            by_sha = {r["sha"]: r for r in plist}
            for j, s in enumerate(shas):
                row = by_sha.get(s)
                if row is None:
                    raise RuntimeError(f"Ensembling: sha {s} missing from run {usable[i]:02d}; all "
                                       "ensembled runs must share the same fixed split.")
                mat[i, j] = float(row["prob"])
        return mat

    val_lists = [load_json(pred_dir / f"val_run_{i:02d}.json") for i in usable]
    val_shas = [r["sha"] for r in val_lists[0]]
    yv = np.asarray([int(r["label"]) for r in val_lists[0]], dtype=int)
    val_mat = _matrix(val_lists, val_shas)

    test_lists = [load_json(pred_dir / f"run_{i:02d}.json") for i in usable]
    test_shas = [r["sha"] for r in test_lists[0]]
    yt = np.asarray([int(r["label"]) for r in test_lists[0]], dtype=int)
    test_projects = [str(r.get("project", "")) for r in test_lists[0]]
    et = np.asarray([float(r.get("effort", 1.0)) for r in test_lists[0]], dtype=float)
    test_mat = _matrix(test_lists, test_shas)

    return usable, yv, val_mat, test_shas, yt, test_projects, et, test_mat


def _write_ensemble_summary(output_dir: Path, completed_run_indices: Sequence[int]) -> None:
    """Bag completed runs' predictions and, if it wins on validation, also fit a
    logistic-regression stacker over the per-run probabilities as a nonlinear
    alternative to the plain average. Both are reported; neither touches test
    labels for model/weight/threshold selection.
    """
    pred_dir = output_dir / "predictions"
    ens = _load_ensemble_matrices(pred_dir, completed_run_indices)
    if ens is None:
        print("[ensemble] fewer than 2 runs have both val/test prediction files yet; skipping for now.")
        return
    usable, yv_ens, val_mat, test_shas, yt_ens, test_projects, et_ens, test_mat = ens

    pv_bag = val_mat.mean(axis=0)
    pt_bag = test_mat.mean(axis=0)
    thr_bag, f1_bag_val = best_f1_threshold(yv_ens, pv_bag)
    auc_bag_val = binary_auc(yv_ens, pv_bag)
    bag_metrics = all_metrics(yt_ens, pt_bag, et_ens, thr_bag)
    rows: List[Dict[str, Any]] = [{
        "dataset": "ALL_21_PROJECTS", "method": "bagged_mean",
        "n_runs_ensembled": len(usable), "threshold": float(thr_bag), **bag_metrics,
    }]
    best_pt = pt_bag

    try:
        from sklearn.linear_model import LogisticRegression
        clf = LogisticRegression(max_iter=2000, C=1.0)
        clf.fit(val_mat.T, yv_ens)
        pv_stack = clf.predict_proba(val_mat.T)[:, 1]
        pt_stack = clf.predict_proba(test_mat.T)[:, 1]
        thr_stack, f1_stack_val = best_f1_threshold(yv_ens, pv_stack)
        auc_stack_val = binary_auc(yv_ens, pv_stack)
        if (f1_stack_val + auc_stack_val) > (f1_bag_val + auc_bag_val):
            stack_metrics = all_metrics(yt_ens, pt_stack, et_ens, thr_stack)
            rows.append({
                "dataset": "ALL_21_PROJECTS", "method": "logreg_stack_over_runs",
                "n_runs_ensembled": len(usable), "threshold": float(thr_stack), **stack_metrics,
            })
            best_pt = pt_stack
    except Exception as ex:
        print(f"[ensemble] stacker skipped: {ex}")

    _atomic_csv_write(rows, output_dir / "ensemble_summary.csv", list(rows[0].keys()))
    _atomic_json_dump(
        [{"sha": s, "project": p, "label": int(y), "prob": float(pb)}
         for s, p, y, pb in zip(test_shas, test_projects, yt_ens, best_pt)],
        pred_dir / "ensemble_best_test.json",
    )
    for row in rows:
        print(f"[ensemble] {row['method']} over {row['n_runs_ensembled']} runs: "
              f"F1={row['F1']:.4f} AUC={row['AUC']:.4f} MCC={row['MCC']:.4f} thr={row['threshold']:.4f}")


def cmd_run(args: argparse.Namespace) -> bool:
    cache_dir = Path(args.cache_dir).resolve()
    output_dir = ensure_dir(Path(args.output_dir).resolve())
    deadline = getattr(args, "session_deadline", None)
    cfg, features = load_commit_features(cache_dir)
    torch, _, _, _, _ = _torch_modules()
    device = args.device
    if device == "auto":
        device = "cuda" if torch.cuda.is_available() else "cpu"
    if device == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("DEVICE='cuda' but this Colab runtime has no CUDA GPU. Enable a GPU runtime first.")
    print(f"[run] device={device}")

    split_seed = int(getattr(args, "split_seed", 42))
    stratify = bool(getattr(args, "stratify_split", False))
    train, val, test = global_random_split(features, seed=split_seed, stratify=stratify)
    if len(train) < 2 or len(val) < 1 or len(test) < 1:
        raise RuntimeError("Whole-dataset 8:1:1 split produced an empty partition.")

    print(f"[run] WHOLE DATASET: total={len(features):,} projects={len(set(x.project for x in features))}")
    print(f"[run] split=8:1:1 random seed={split_seed} stratified={stratify}")
    print(f"[run] train={len(train):,} ({sum(x.label for x in train):,} defective) | "
          f"val={len(val):,} ({sum(x.label for x in val):,} defective) | "
          f"test={len(test):,} ({sum(x.label for x in test):,} defective)")

    split_rows = [{
        "dataset": "ALL_21_PROJECTS", "total": len(features),
        "defective": int(sum(x.label for x in features)),
        "clean": int(len(features) - sum(x.label for x in features)),
        "train": len(train), "train_defective": int(sum(x.label for x in train)),
        "validation": len(val), "validation_defective": int(sum(x.label for x in val)),
        "test": len(test), "test_defective": int(sum(x.label for x in test)),
        "split_seed": split_seed, "stratified": stratify,
    }]
    _atomic_csv_write(split_rows, output_dir / "dataset_splits.csv", list(split_rows[0].keys()))

    manifest_rows: List[Dict[str, Any]] = []
    for split_name, items in (("train", train), ("validation", val), ("test", test)):
        for cf in items:
            manifest_rows.append({"split": split_name, "project": cf.project, "sha": cf.sha,
                                  "label": cf.label, "author_date": cf.author_date, "effort": cf.effort})
    _atomic_csv_write(manifest_rows, output_dir / "global_split_manifest.csv", list(manifest_rows[0].keys()))

    # Project composition is descriptive only; the model is never trained per project.
    composition: List[Dict[str, Any]] = []
    for p in sorted({x.project for x in features}):
        row: Dict[str, Any] = {"project": p}
        for split_name, items in (("train", train), ("validation", val), ("test", test)):
            xx = [x for x in items if x.project == p]
            row[f"{split_name}_n"] = len(xx)
            row[f"{split_name}_defective"] = int(sum(x.label for x in xx))
        composition.append(row)
    _atomic_csv_write(composition, output_dir / "split_by_project.csv", list(composition[0].keys()))

    booster_val_prob = booster_test_prob = None
    if bool(globals().get("USE_THREE_VIEW_BOOSTER", True)):
        booster_val_prob, booster_test_prob = _build_three_view_booster_predictions(
            train, val, test, output_dir, device, int(globals().get("HASH_FEATURES", 4096))
        )

    definitions = {
        "protocol": "One global 21-project model; fixed random 8:1:1 split. Neural FUSE-JIT is complemented by a validation-tuned XGBoost classifier built from the same three views. Blend weight and threshold are selected on validation only.",
        "Prec": "TP/(TP+FP) at validation-selected F1 threshold.",
        "Recall": "TP/(TP+FN) at validation-selected F1 threshold.",
        "AUC": "ROC AUC from test probabilities.",
        "MCC": "Matthews correlation coefficient.",
        "Accuracy": "(TP+TN)/N.",
        "Balanced_Accuracy": "mean of sensitivity and specificity.",
        "R@20%E": "Defective-commit recall within 20% of total changed-line effort, ranked by predicted defect density prob/effort.",
        "PofB20": "Alias of R@20%E retained for compatibility with the original script.",
        "E@20%R": "Fraction of effort required to find 20% of defective commits under predicted defect-density ranking.",
        "Popt": "Normalized area between predicted, optimal, and worst effort-recall curves.",
        "Top5": "COMMIT-level recall among the top 5 commits ranked by predicted defect density.",
        "Top10": "COMMIT-level recall among the top 10 commits ranked by predicted defect density.",
        "IFA": "COMMIT-level initial false alarms: clean commits before the first defective commit in the defect-density ranking.",
        "localization_note": "JIT-Smart reports Top-5/Top-10/IFA primarily for line-level localization. This FUSE-JIT cache has commit labels only, so Top5/Top10/IFA here are explicitly commit-ranking metrics and are not claimed to be line-localization metrics.",
    }
    _atomic_json_dump(definitions, output_dir / "metric_definitions.json")

    def _print_all_metrics(row: Mapping[str, Any], recovered: bool = False) -> None:
        """Print every requested evaluation metric for one global run."""
        run_no = int(float(row.get("run", 0)))
        prefix = "recovered run" if recovered else "run"
        print(
            f"  {prefix} {run_no:02d}: "
            f"Prec={float(row['Prec']):.4f} "
            f"Recall={float(row['Recall']):.4f} "
            f"F1={float(row['F1']):.4f} "
            f"PofB20={float(row['PofB20']):.4f} "
            f"AUC={float(row['AUC']):.4f} "
            f"MCC={float(row['MCC']):.4f} "
            f"Accuracy={float(row['Accuracy']):.4f} "
            f"Balanced_Accuracy={float(row['Balanced_Accuracy']):.4f} "
            f"R@20%E={float(row['R@20%E']):.4f} "
            f"E@20%R={float(row['E@20%R']):.4f} "
            f"Popt={float(row['Popt']):.4f} "
            f"Top10={float(row['Top10']):.4f} "
            f"Top5={float(row['Top5']):.4f} "
            f"IFA={float(row['IFA']):.4f} "
            f"thr={float(row.get('threshold', 0.0)):.4f} "
            f"epoch={int(float(row.get('best_epoch', 0)))} "
            f"neural_w={float(row.get('neural_blend_weight', 1.0)):.2f} "
            f"val_neural_F1={float(row.get('neural_val_f1', float('nan'))):.4f} "
            f"val_blend_F1={float(row.get('blend_val_f1', float('nan'))):.4f}"
        )

    runs_path = output_dir / "runs.csv"
    all_rows: List[Dict[str, Any]] = _load_existing_run_rows(runs_path)
    completed_runs = {int(r["run"]) for r in all_rows if "run" in r}
    if completed_runs:
        print(f"[run] RESUME: recovered {len(completed_runs):,} completed GLOBAL runs from Google Drive")
        for recovered_row in sorted(all_rows, key=lambda x: int(float(x.get("run", 0)))):
            _print_all_metrics(recovered_row, recovered=True)

    per_project_path = output_dir / "per_project_runs.csv"
    per_project_rows = _load_csv_dicts(per_project_path)
    safe_stop = False
    todo = [r for r in range(args.runs) if r + 1 not in completed_runs]
    print(f"[run] global model: completed={len(completed_runs)} remaining={len(todo)}")
    for r in todo:
        if _deadline_reached(deadline):
            safe_stop = True
            break
        result = train_one_run(
            "ALL_21_PROJECTS", train, val, test, cfg, r, args.epochs, args.batch_size,
            args.learning_rate, device,
            weight_decay=float(getattr(args, "weight_decay", 1e-4)),
            patience=int(getattr(args, "early_stop_patience", 15)),
            grad_clip=float(getattr(args, "grad_clip", 1.0)),
            booster_val_prob=booster_val_prob, booster_test_prob=booster_test_prob,
            blend_grid_points=int(globals().get("BOOSTER_BLEND_GRID", 41)),
            contrastive_weight=float(getattr(args, "contrastive_weight", 0.15)),
            contrastive_temperature=float(getattr(args, "contrastive_temperature", 0.1)),
        )
        pred = result.pop("test_predictions")
        val_pred = result.pop("val_predictions")
        project_metrics = result.pop("per_project_metrics")
        _atomic_json_dump(pred, output_dir / "predictions" / f"run_{r+1:02d}.json")
        _atomic_json_dump(val_pred, output_dir / "predictions" / f"val_run_{r+1:02d}.json")
        all_rows.append(result)
        completed_runs.add(r + 1)
        # Remove any stale subgroup rows for this run, then append current diagnostics.
        per_project_rows = [x for x in per_project_rows if int(float(x.get("run", -1))) != r + 1]
        per_project_rows.extend(project_metrics)
        _atomic_csv_write(all_rows, runs_path, list(all_rows[0].keys()))
        if per_project_rows:
            fields = list(per_project_rows[0].keys())
            _atomic_csv_write(per_project_rows, per_project_path, fields)
        _print_all_metrics(result, recovered=False)

    if all_rows:
        summary: List[Dict[str, Any]] = []
        for metric in METRIC_NAMES:
            vals = np.asarray([float(r[metric]) for r in all_rows], dtype=float)
            summary.append({
                "dataset": "ALL_21_PROJECTS", "metric": metric,
                "mean": float(vals.mean()),
                "sample_sd": float(vals.std(ddof=1)) if len(vals) > 1 else 0.0,
                "n_runs": len(vals),
            })
        _atomic_csv_write(summary, output_dir / "summary.csv", list(summary[0].keys()))

    if per_project_rows:
        psummary: List[Dict[str, Any]] = []
        for p in sorted({str(x["project"]) for x in per_project_rows}):
            rs = [x for x in per_project_rows if str(x["project"]) == p]
            for metric in METRIC_NAMES:
                vals = np.asarray([float(x[metric]) for x in rs], dtype=float)
                psummary.append({"project": p, "metric": metric, "mean": float(vals.mean()),
                                 "sample_sd": float(vals.std(ddof=1)) if len(vals) > 1 else 0.0,
                                 "n_runs": len(vals)})
        _atomic_csv_write(psummary, output_dir / "per_project_summary.csv", list(psummary[0].keys()))

    _write_ensemble_summary(output_dir, sorted(completed_runs))

    if safe_stop:
        print(f"[run] SAFE STOP: {len(completed_runs):,}/{args.runs} global runs are persisted on Google Drive.")
        print("[run] Start a new Colab runtime and run the SAME script; completed global runs will be skipped.")
        return False
    print(f"[run] global results written persistently to {output_dir}")
    return True


def build_arg_parser() -> argparse.ArgumentParser:
    p = argparse.ArgumentParser(formatter_class=argparse.ArgumentDefaultsHelpFormatter)
    sub = p.add_subparsers(dest="cmd", required=True)

    a = sub.add_parser("prepare", help="Map JIT-Defect-Extended labels to Git commits and reconstruct old/new pairs + process metrics")
    a.add_argument("--dataset-root", required=True, help="Path to JIT-Defect-Extended/dataset")
    a.add_argument("--repos-root", required=True, help="Directory containing the 21 Git repositories")
    a.add_argument("--cache-dir", required=True)
    a.add_argument("--clone-repos", action="store_true", help="Clone missing Apache repositories")
    a.add_argument("--labels-csv", default=None, help="Optional explicit CSV with sha,label[,project] instead of automatic dataset discovery")
    a.add_argument("--max-files", type=int, default=100, help="Benchmark-style extreme-change guard")
    a.add_argument("--max-changed-loc", type=int, default=10000, help="Benchmark-style extreme-change guard")
    a.set_defaults(func=cmd_prepare)

    f = sub.add_parser("featurize", help="Extract AST syntax changes, hybrid graph changes, Word/Node2Vec-ready data")
    f.add_argument("--cache-dir", required=True)
    f.add_argument("--syntax-dim", type=int, default=100)
    f.add_argument("--syntax-window", type=int, default=10)
    f.add_argument("--seq-len", type=int, default=50)
    f.add_argument("--graph-dim", type=int, default=100)
    f.add_argument("--graph-window", type=int, default=5)
    f.add_argument("--walk-length", type=int, default=20)
    f.add_argument("--num-walks", type=int, default=10)
    f.add_argument("--node2vec-p", type=float, default=1.0)
    f.add_argument("--node2vec-q", type=float, default=1.0)
    f.set_defaults(func=cmd_featurize)

    r = sub.add_parser("run", help="Run one whole-dataset FUSE-JIT experiment suite")
    r.add_argument("--cache-dir", required=True)
    r.add_argument("--output-dir", required=True)
    r.add_argument("--projects", default="", help="Comma-separated project subset; empty = all")
    r.add_argument("--runs", type=int, default=30)
    r.add_argument("--epochs", type=int, default=100)
    r.add_argument("--batch-size", type=int, default=64)
    r.add_argument("--learning-rate", type=float, default=0.001)
    r.add_argument("--device", default="auto", choices=["auto", "cpu", "cuda"])
    r.add_argument("--split-seed", type=int, default=42)
    r.add_argument("--stratify-split", action="store_true")
    r.add_argument("--weight-decay", type=float, default=1e-4)
    r.add_argument("--early-stop-patience", type=int, default=15)
    r.add_argument("--grad-clip", type=float, default=1.0)
    r.add_argument("--contrastive-weight", type=float, default=0.15,
                   help="Weight of the auxiliary supervised-contrastive loss on the fused embedding; 0 disables it")
    r.add_argument("--contrastive-temperature", type=float, default=0.1)
    r.set_defaults(func=cmd_run)
    return p


def main(argv: Optional[Sequence[str]] = None) -> None:
    p = build_arg_parser()
    args = p.parse_args(argv)
    args.func(args)


# ============================================================================
# GOOGLE COLAB DIRECT-RUNNER -- PERSISTENT / RESUMABLE
# ============================================================================
# Scientific mechanisms and parameters above are unchanged. This runner adds
# Google Drive persistence, atomic checkpoints, safe-session stopping, and resume.

from types import SimpleNamespace

# ----------------------------- USER CONFIG ---------------------------------

# Fast ephemeral workspace for active computation.
WORK_DIR = Path("/content/FUSE_JIT_WORK")
DATASET_REPO_DIR = WORK_DIR / "JIT-Defect-Extended"
DATASET_ROOT = DATASET_REPO_DIR / "dataset"
DATASET_URL = "https://github.com/JIT-LSM/JIT-Defect-Extended.git"
REPOS_ROOT = WORK_DIR / "repositories"
CACHE_DIR = WORK_DIR / "cache"

# Persistent Google Drive storage. Do NOT point Git repositories here: Git and
# Tree-sitter work much faster on /content. Only checkpoints/final caches/results
# are persisted to Drive.
DRIVE_MOUNT = Path("/content/drive")
PERSIST_ROOT = DRIVE_MOUNT / "MyDrive" / "FUSE_JIT_PERSISTENT"
PERSIST_CACHE_DIR = PERSIST_ROOT / "cache"
OUTPUT_DIR = PERSIST_ROOT / "results_validation_tuned_ensemble"

# Safety policy: persistent Stage-1/Stage-2 checkpoint shards are NEVER deleted.
# This uses extra Drive space but guarantees recovery even if a final cache file
# is accidentally removed or becomes unavailable.
KEEP_PERSISTENT_CHECKPOINT_SHARDS = True

RUN_PREPARE = True
RUN_FEATURIZE = True
RUN_SEMANTIC = True
RUN_TRAIN = True
FORCE_PREPARE = False
FORCE_FEATURIZE = False
FORCE_SEMANTIC = False
AUTO_CLONE_DATASET = True
AUTO_CLONE_PROJECT_REPOS = True

# Stage 2.5: pretrained code-model embedding of each commit's diff (see
# build_change_diff_text / extract_semantic_embeddings). This is the single
# highest-leverage addition toward the paper-level F1/AUC targets: it replaces
# nothing but adds a view carrying pretrained code understanding that a small
# from-scratch Word2Vec model trained on ~22K commits cannot match. Runs ONCE
# for the whole dataset with its own resumable checkpointing, independent of
# Stage 2 and of the 30 Stage-3 training runs.
USE_SEMANTIC_EMBEDDINGS = True
SEMANTIC_MODEL_NAME = "microsoft/codebert-base"
SEMANTIC_MAX_LENGTH = 256
SEMANTIC_BATCH_SIZE = 64
SEMANTIC_MAX_DIFF_CHARS = 6000
SEMANTIC_USE_FP16 = True
SEMANTIC_CHECKPOINT_EVERY = 2000

# Parallel CPU scheduling only; scientific feature extraction is unchanged.
COLAB_CPU_COUNT = max(1, os.cpu_count() or 1)
PREPARE_WORKERS = max(1, min(8, COLAB_CPU_COUNT))
# Stage 2 worker target. Set to 64 if your runtime actually provides >=64 CPUs.
FEATURIZE_WORKER_TARGET = 64
FEATURIZE_WORKERS = max(1, min(FEATURIZE_WORKER_TARGET, COLAB_CPU_COUNT))

# Persistent checkpoint cadence. A hard disconnect loses at most the current batch.
PREPARE_CHECKPOINT_EVERY = 200
FEATURIZE_CHECKPOINT_EVERY = 100

# Colab disconnected around 12h in the reported run. Stop cleanly before that,
# persist state, then continue in a fresh runtime by rerunning this same file.
SAFE_SESSION_HOURS = 10.0

PROJECTS = ""  # retained only for backward CLI compatibility; Stage 3 uses ALL projects.

# Whole-dataset Stage-3 protocol.
SPLIT_SEED = 42
STRATIFY_SPLIT = False  # keep the same fixed split as the stronger previous model for an apples-to-apples comparison.
WEIGHT_DECAY = 1e-4
EARLY_STOP_PATIENCE = 15
GRAD_CLIP = 1.0
# Auxiliary supervised-contrastive loss weight/temperature on the fused embedding.
# Set CONTRASTIVE_WEIGHT = 0.0 to fully recover the previous (non-contrastive) model.
CONTRASTIVE_WEIGHT = 0.15
CONTRASTIVE_TEMPERATURE = 0.1

# FUSE-JIT experiment settings.
RUNS = 30
EPOCHS = 100
BATCH_SIZE = 64
LEARNING_RATE = 0.001
DEVICE = "cuda"

# Validation-tuned complementary three-view booster. It uses ONLY the same
# process + syntactic + graph views and does not touch test labels during tuning.
USE_THREE_VIEW_BOOSTER = True
HASH_FEATURES = 4096
BOOSTER_BLEND_GRID = 41   # alpha in [0,1]; alpha=1 => neural only
BOOSTER_CONFIGS = [
    {"max_depth": 4, "min_child_weight": 3, "learning_rate": 0.035, "subsample": 0.90, "colsample_bytree": 0.90, "scale_mult": 0.75},
    {"max_depth": 5, "min_child_weight": 5, "learning_rate": 0.030, "subsample": 0.90, "colsample_bytree": 0.85, "scale_mult": 0.75},
    {"max_depth": 4, "min_child_weight": 5, "learning_rate": 0.025, "subsample": 0.85, "colsample_bytree": 0.90, "scale_mult": 1.00},
    {"max_depth": 6, "min_child_weight": 8, "learning_rate": 0.025, "subsample": 0.85, "colsample_bytree": 0.80, "scale_mult": 0.60},
    # Wider search added alongside the AGE/EXP/REXP/SEXP expert metrics: the extra
    # numeric signal benefits from a couple of deeper/slower and shallower/faster
    # points that the original 4-config grid did not cover. The booster is trained
    # once (not per neural run), so widening this grid is a one-time cost.
    {"max_depth": 7, "min_child_weight": 10, "learning_rate": 0.015, "subsample": 0.80, "colsample_bytree": 0.75, "scale_mult": 0.65},
    {"max_depth": 3, "min_child_weight": 2, "learning_rate": 0.05, "subsample": 0.95, "colsample_bytree": 0.95, "scale_mult": 0.90},
    {"max_depth": 5, "min_child_weight": 4, "learning_rate": 0.02, "subsample": 0.85, "colsample_bytree": 0.85, "scale_mult": 1.10},
]


# Feature settings -- UNCHANGED.
SYNTAX_DIM = 100
SYNTAX_WINDOW = 10
SEQ_LEN = 50
GRAPH_DIM = 100
GRAPH_WINDOW = 5
WALK_LENGTH = 20
NUM_WALKS = 10
NODE2VEC_P = 1.0
NODE2VEC_Q = 1.0
MAX_FILES = 100
MAX_CHANGED_LOC = 10000
LABELS_CSV = None

# --------------------------- END CONFIG ------------------------------------


def banner(text: str) -> None:
    print("\n" + "=" * 88)
    print(text)
    print("=" * 88)


def ensure_colab_dependencies() -> None:
    required = [
        ("torch", "torch"),
        ("gensim", "gensim"),
        ("tree_sitter", "tree-sitter"),
        ("tree_sitter_java", "tree-sitter-java"),
        ("sklearn", "scikit-learn"),
        ("scipy", "scipy"),
        ("xgboost", "xgboost"),
        ("transformers", "transformers"),
    ]
    missing = []
    for module_name, pip_name in required:
        try:
            __import__(module_name)
        except Exception:
            missing.append(pip_name)
    if missing:
        banner("Installing missing Python dependencies")
        print("[environment] pip install", " ".join(missing))
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    for module_name, _ in required:
        __import__(module_name)


def ensure_git() -> None:
    if shutil.which("git") is None:
        raise RuntimeError("Git is not available in this Colab runtime.")
    p = subprocess.run(["git", "--version"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, check=False)
    print("[environment]", p.stdout.strip() or p.stderr.strip())


def ensure_google_drive() -> None:
    try:
        from google.colab import drive
    except Exception as ex:
        raise RuntimeError("This persistent version is intended for Google Colab and requires google.colab.drive.") from ex
    banner("Mounting Google Drive for persistent checkpoints")
    drive.mount(str(DRIVE_MOUNT), force_remount=False)
    ensure_dir(PERSIST_CACHE_DIR)
    ensure_dir(OUTPUT_DIR)
    print("[persistent]", PERSIST_ROOT)


def restore_persistent_final(local_path: Path, persistent_path: Path) -> bool:
    if local_path.exists():
        return True
    if persistent_path.exists():
        ensure_dir(local_path.parent)
        print(f"[restore] Google Drive -> local: {persistent_path.name}")
        shutil.copy2(persistent_path, local_path)
        return True
    return False


def _commit_records_schema_ok(path: Path) -> bool:
    """Whether a cached commit_records.pkl already has every current process metric.

    Lets a Colab rerun self-heal after a process-metric schema upgrade (e.g. adding
    AGE/EXP/REXP/SEXP) without the user having to manually clear Google Drive state.
    """
    try:
        with path.open("rb") as f:
            records = pickle.load(f)
        if not records:
            return True
        r0 = records[0]
        proc = r0.process if isinstance(r0, CommitRecord) else dict(r0.get("process", {}))
        return set(PROCESS_FEATURES).issubset(set(proc.keys()))
    except Exception:
        return False


def _commit_features_schema_ok(path: Path) -> bool:
    """Whether a cached commit_features.pkl's process vectors match PROCESS_FEATURES."""
    try:
        with path.open("rb") as f:
            obj = pickle.load(f)
        feats = obj["features"] if isinstance(obj, dict) and "features" in obj else obj
        if not feats:
            return True
        return int(np.asarray(feats[0].process).shape[0]) == len(PROCESS_FEATURES)
    except Exception:
        return False


def _refresh_process_metrics_if_possible(local_features_path: Path, local_records_path: Path) -> bool:
    """Patch process vectors in an existing commit_features.pkl from a freshly
    regenerated commit_records.pkl, without re-running the expensive tree-sitter/
    Node2Vec extraction, when only the process-metric schema changed (the AST/graph
    syntax_tokens and graph_tokens per commit are untouched by that schema and stay
    100% reusable). Returns True if the patch was applied; False means a full Stage 2
    rerun is required (e.g. the underlying commit set diverged).
    """
    if _commit_features_schema_ok(local_features_path):
        return False
    try:
        with local_features_path.open("rb") as f:
            obj = pickle.load(f)
        is_wrapped = isinstance(obj, dict) and "features" in obj
        feats: List[CommitFeatures] = obj["features"] if is_wrapped else obj
        records = load_commit_records(local_records_path.parent)
        by_key = {(r.project, r.sha): r for r in records}
        if len(by_key) != len(records) or not feats:
            return False
        for cf in feats:
            r = by_key.get((cf.project, cf.sha))
            if r is None or not set(PROCESS_FEATURES).issubset(set(r.process.keys())):
                return False
            cf.process = np.asarray([r.process[k] for k in PROCESS_FEATURES], dtype=np.float32)
            cf.effort = max(1.0, float(r.process["LA"] + r.process["LD"]))
        new_obj = {"config": obj["config"], "features": feats} if is_wrapped else feats
        _atomic_pickle_dump(new_obj, local_features_path)
        print(f"[features] refreshed process metrics in place for {len(feats):,} cached commits "
              "(syntax/graph features reused; no AST/Node2Vec recomputation needed)")
        return True
    except Exception as ex:
        print(f"[features] in-place process-metric refresh failed, falling back to full re-extraction: {ex}")
        return False


def _commit_features_semantic_ok(path: Path) -> bool:
    """Whether every cached CommitFeatures already carries a semantic_emb."""
    try:
        with path.open("rb") as f:
            obj = pickle.load(f)
        feats = obj["features"] if isinstance(obj, dict) and "features" in obj else obj
        if not feats:
            return True
        return all(getattr(cf, "semantic_emb", None) is not None for cf in feats)
    except Exception:
        return False


def ensure_dataset() -> None:
    if DATASET_ROOT.exists():
        print(f"[dataset] using existing dataset: {DATASET_ROOT}")
        return
    if not AUTO_CLONE_DATASET:
        raise FileNotFoundError(f"Dataset not found: {DATASET_ROOT}")
    DATASET_REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    if DATASET_REPO_DIR.exists() and not (DATASET_REPO_DIR / ".git").exists():
        raise RuntimeError(f"{DATASET_REPO_DIR} exists but is not a Git clone.")
    if not DATASET_REPO_DIR.exists():
        banner("Downloading JIT-Defect-Extended")
        subprocess.run(["git", "clone", DATASET_URL, str(DATASET_REPO_DIR)], check=True)
    if not DATASET_ROOT.exists():
        raise FileNotFoundError(f"Expected dataset directory not found: {DATASET_ROOT}")


def run_prepare_colab(session_deadline: float) -> bool:
    local_final = CACHE_DIR / "commit_records.pkl"
    persistent_final = PERSIST_CACHE_DIR / "commit_records.pkl"
    if not FORCE_PREPARE and restore_persistent_final(local_final, persistent_final):
        if _commit_records_schema_ok(local_final):
            print(f"[prepare] persistent final cache exists; skipping reconstruction: {persistent_final}")
            return True
        print(f"[prepare] cached commit_records.pkl predates the current process-metric schema "
              f"{PROCESS_FEATURES}; re-deriving Stage 1 to add the missing metrics.")

    ensure_dataset()
    banner("STAGE 1/3 - Reconstruct commit-level dataset")
    args = SimpleNamespace(
        dataset_root=str(DATASET_ROOT), repos_root=str(REPOS_ROOT), cache_dir=str(CACHE_DIR),
        persistent_cache_dir=str(PERSIST_CACHE_DIR), clone_repos=AUTO_CLONE_PROJECT_REPOS,
        labels_csv=str(LABELS_CSV) if LABELS_CSV else None,
        max_files=MAX_FILES, max_changed_loc=MAX_CHANGED_LOC,
        prepare_workers=PREPARE_WORKERS, checkpoint_every=PREPARE_CHECKPOINT_EVERY,
        session_deadline=session_deadline,
    )
    return bool(cmd_prepare(args))


def run_featurize_colab(session_deadline: float) -> bool:
    local_final = CACHE_DIR / "commit_features.pkl"
    persistent_final = PERSIST_CACHE_DIR / "commit_features.pkl"
    local_records = CACHE_DIR / "commit_records.pkl"
    if not FORCE_FEATURIZE:
        restore_persistent_final(local_final, persistent_final)
        if local_final.exists() and not _commit_features_schema_ok(local_final) and local_records.exists():
            if _refresh_process_metrics_if_possible(local_final, local_records):
                _atomic_copy_file(local_final, persistent_final)
        if local_final.exists() and _commit_features_schema_ok(local_final):
            print(f"[featurize] persistent final cache exists; skipping extraction: {persistent_final}")
            return True
        if local_final.exists():
            print("[featurize] cached commit_features.pkl predates the current process-metric schema "
                  "and could not be patched in place; re-running Stage 2 feature extraction.")
    if not (CACHE_DIR / "commit_records.pkl").exists():
        raise FileNotFoundError("Missing commit_records.pkl; Stage 1 must complete first.")

    banner("STAGE 2/3 - Extract FUSE-JIT syntax and graph features")
    args = SimpleNamespace(
        cache_dir=str(CACHE_DIR), persistent_cache_dir=str(PERSIST_CACHE_DIR),
        syntax_dim=SYNTAX_DIM, syntax_window=SYNTAX_WINDOW, seq_len=SEQ_LEN,
        graph_dim=GRAPH_DIM, graph_window=GRAPH_WINDOW,
        walk_length=WALK_LENGTH, num_walks=NUM_WALKS,
        node2vec_p=NODE2VEC_P, node2vec_q=NODE2VEC_Q,
        featurize_workers=FEATURIZE_WORKERS, checkpoint_every=FEATURIZE_CHECKPOINT_EVERY,
        session_deadline=session_deadline,
    )
    return bool(cmd_featurize(args))


def run_semantic_colab(session_deadline: float) -> bool:
    local_features = CACHE_DIR / "commit_features.pkl"
    persistent_features = PERSIST_CACHE_DIR / "commit_features.pkl"
    if not local_features.exists():
        raise FileNotFoundError("Missing commit_features.pkl; Stage 2 must complete first.")
    if not USE_SEMANTIC_EMBEDDINGS:
        print("[semantic] USE_SEMANTIC_EMBEDDINGS is False; skipping the pretrained-embedding stage.")
        return True
    if not FORCE_SEMANTIC and _commit_features_semantic_ok(local_features):
        print("[semantic] every cached commit already has a semantic embedding; skipping.")
        return True

    banner("STAGE 2.5/3 - Extract pretrained semantic embeddings")
    torch, _, _, _, _ = _torch_modules()
    device = DEVICE
    if device == "auto":
        device = "cuda" if torch.cuda.is_available() else "cpu"
    records = load_commit_records(CACHE_DIR)
    cfg, features = load_commit_features(CACHE_DIR)
    finished = extract_semantic_embeddings(
        records, features, stage_root=PERSIST_CACHE_DIR, device=device,
        model_name=SEMANTIC_MODEL_NAME, max_length=SEMANTIC_MAX_LENGTH,
        batch_size=SEMANTIC_BATCH_SIZE, max_diff_chars=SEMANTIC_MAX_DIFF_CHARS,
        use_fp16=SEMANTIC_USE_FP16, checkpoint_every=SEMANTIC_CHECKPOINT_EVERY,
        session_deadline=session_deadline,
    )
    final_obj = {"config": dataclasses.asdict(cfg), "features": features}
    _atomic_pickle_dump(final_obj, local_features)
    _atomic_pickle_dump(final_obj, persistent_features)
    return finished


def run_training_colab(session_deadline: float) -> bool:
    if not (CACHE_DIR / "commit_features.pkl").exists():
        raise FileNotFoundError("Missing commit_features.pkl; Stage 2 must complete first.")
    banner("STAGE 3/3 - Train and evaluate FUSE-JIT")
    args = SimpleNamespace(
        cache_dir=str(CACHE_DIR), output_dir=str(OUTPUT_DIR), projects=PROJECTS,
        runs=RUNS, epochs=EPOCHS, batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE, device=DEVICE,
        split_seed=SPLIT_SEED, stratify_split=STRATIFY_SPLIT,
        weight_decay=WEIGHT_DECAY, early_stop_patience=EARLY_STOP_PATIENCE,
        grad_clip=GRAD_CLIP, session_deadline=session_deadline,
        contrastive_weight=CONTRASTIVE_WEIGHT, contrastive_temperature=CONTRASTIVE_TEMPERATURE,
    )
    return bool(cmd_run(args))


def show_resume_message(stage: str) -> None:
    banner("PERSISTENT CHECKPOINT SAVED - SAFE TO END THIS COLAB SESSION")
    print(f"{stage} is not finished yet, but completed work is stored in Google Drive:")
    print(" ", PERSIST_ROOT)
    print("Open a fresh Colab runtime, enable GPU, upload/run this SAME Python file, and it will resume.")
    print("Do NOT delete the FUSE_JIT_PERSISTENT folder in Google Drive.")


def show_outputs_colab() -> None:
    banner("FINISHED")
    print("Fast local workspace :", WORK_DIR)
    print("Persistent workspace :", PERSIST_ROOT)
    print("Persistent cache     :", PERSIST_CACHE_DIR)
    print("Persistent results   :", OUTPUT_DIR)
    print()
    print("Results:")
    print("  ", OUTPUT_DIR / "dataset_splits.csv")
    print("  ", OUTPUT_DIR / "global_split_manifest.csv")
    print("  ", OUTPUT_DIR / "runs.csv")
    print("  ", OUTPUT_DIR / "summary.csv")
    print("  ", OUTPUT_DIR / "per_project_runs.csv")
    print("  ", OUTPUT_DIR / "per_project_summary.csv")
    print("  ", OUTPUT_DIR / "ensemble_summary.csv")
    print("  ", OUTPUT_DIR / "metric_definitions.json")
    print("  ", OUTPUT_DIR / "predictions")


def colab_main() -> None:
    session_start = time.time()
    session_deadline = session_start + SAFE_SESSION_HOURS * 3600.0
    ensure_git()
    ensure_colab_dependencies()
    ensure_google_drive()
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    CACHE_DIR.mkdir(parents=True, exist_ok=True)

    # Restore expensive final caches first. If Stage 1 is already persisted,
    # subsequent sessions do not need to clone the 21 Git repositories at all.
    restore_persistent_final(CACHE_DIR / "commit_records.pkl", PERSIST_CACHE_DIR / "commit_records.pkl")
    restore_persistent_final(CACHE_DIR / "commit_features.pkl", PERSIST_CACHE_DIR / "commit_features.pkl")

    if RUN_PREPARE:
        if not run_prepare_colab(session_deadline):
            show_resume_message("Stage 1")
            return
    if RUN_FEATURIZE:
        if not run_featurize_colab(session_deadline):
            show_resume_message("Stage 2")
            return
    if RUN_SEMANTIC:
        if not run_semantic_colab(session_deadline):
            show_resume_message("Stage 2.5")
            return
    if RUN_TRAIN:
        if not run_training_colab(session_deadline):
            show_resume_message("Stage 3")
            return
    show_outputs_colab()


if __name__ == "__main__":
    colab_main()
